# Trabalho Prático — Algoritmos de Ordenação
### Estrutura de Dados · 4º período

---

| | |
|---|---|
| **Aluno(s):** | Arthur Silva Teixeira, Rafael Campello Soares |
| **Matrícula(s):** | 2024021870, 2024017457 |
| **Data de entrega:** | 29/10/2026 |


---

Este notebook é, ao mesmo tempo, **o enunciado**, **o material de apoio** e **o seu relatório**. Você vai lê-lo de cima para baixo, estudar a teoria de cada algoritmo, implementar o que está marcado com `TODO`, rodar os experimentos e escrever suas análises nas células marcadas com ✍️.

> **Como ler as marcações deste notebook**
> - 📘 **Teoria** — leia com atenção; é a base para as suas análises.
> - 🧰 **Código de apoio** — já está pronto. Execute, leia e entenda, mas **não altere** a menos que seja pedido.
> - 🛠️ **TODO** — você deve implementar.
> - ✍️ **Resposta do aluno** — você deve escrever (texto, fórmulas, tabelas).
> - ⚠️ **Atenção** — armadilhas comuns de iniciantes.

## Sumário

1. [Parte A — Algoritmos quadráticos](#parteA)
2. [Parte B — Algoritmos subquadráticos (Shellsort, Merge Sort, Quicksort)](#parteB)
3. [Parte C — Algoritmos de tempo linear (Counting, Bucket, Radix)](#parteC)
4. [Parte D — Verificação de corretude](#parteD)
5. [Parte E — Análise teórica consolidada](#parteE)
6. [Parte F — Avaliação experimental](#parteF)
7. [Parte G — Discussão dos resultados](#parteG)
8. [Parte H — Relatório](#parteH)

<a id="tarefas"></a>
## 1. Visão geral das tarefas

| Tarefa | O que fazer | Onde no notebook | Peso |
|---|---|---|---|
| **T1 — Implementação** | Implementar os 9 algoritmos, instrumentados para contar comparações e trocas, **sem** usar `sort()`, `sorted()`, `heapq`, `numpy.sort` ou similares. | Partes A, B, C | 25 |
| **T2 — Corretude** | Passar na bateria de testes fornecida e acrescentar pelo menos 3 testes próprios. | Parte D | 5 |
| **T3 — Análise teórica** | Para cada algoritmo: melhor, médio e pior caso (tempo), espaço extra, estabilidade; recorrências de Merge Sort e Quicksort resolvidas pelo Teorema Mestre. | Partes A, B, C, E | 20 |
| **T4 — Avaliação experimental** | Executar os experimentos com as faixas de *n* especificadas, 5 seeds, vários tipos de entrada; gerar tabelas e gráficos (inclusive log-log) comparando com as curvas teóricas. | Parte F | 20 |
| **T5 — Discussão** | Responder às questões de discussão com base **nos seus dados**. | Parte G | 15 |
| **T6 — Relatório** | Introdução, fundamentação, metodologia, resultados, análise crítica, conclusões e referências. | Parte H | 15 |

**Ordem de trabalho recomendada** (não pule etapas!):
1. Leia as seções 2, 3 e 4 e execute as células de apoio.
2. Implemente **um** algoritmo, rode os testes da Parte D para ele e só depois passe ao próximo.
3. Faça a análise teórica de cada algoritmo **logo após** implementá-lo — a implementação ajuda a enxergar os casos.
4. Rode primeiro os experimentos no **modo rápido** (`MODO = "rapido"`) para depurar; só então rode o **modo completo**.
5. Escreva a discussão e o relatório por último, olhando para os gráficos.

Não utlize modelos de IA/LLM para implementação dos códigos.

<a id="regras"></a>
## 2. Regras gerais

1. **Linguagem:** Python 3.10 ou superior. Bibliotecas permitidas: `random`, `time`, `math`, `sys`, `gc`, `statistics`, `numpy` (apenas para gerar números e fazer contas nos gráficos), `pandas`, `matplotlib`.
2. **Proibido** usar qualquer rotina de ordenação pronta (`list.sort`, `sorted`, `numpy.sort`, `pandas.sort_values` **dentro dos algoritmos**, `heapq`, `bisect`...). Você pode usar `sorted()` **somente** nas funções de teste, para gerar o gabarito, e `sort_values` para organizar tabelas de resultados.
3. **Assinatura obrigatória** de todos os algoritmos:
   ```python
   def nome_do_algoritmo(v: list, c: Contador) -> list:
   ```
   - `v` é a lista de entrada; `c` é o contador de operações (Seção 4).
   - A função deve **retornar a lista ordenada** em ordem **não decrescente**. Algoritmos *in-place* podem modificar `v` e retorná-la; os demais podem retornar uma nova lista.
   - Não altere os nomes das funções: o código de testes e experimentos depende deles.
4. **O que contar:**
   - `c.comp` — **comparações entre elementos** da entrada (ex.: `v[j] > chave`). Comparações de índices (`j >= 0`, `i < n`) **não** contam.
   - `c.trocas` — **movimentações de elementos**: cada `swap` conta **1**; cada atribuição que escreve um elemento da entrada em uma posição (deslocamento no Insertion Sort, cópia no Merge, escrita no vetor de saída do Counting Sort) também conta **1**.
   - Para algoritmos não comparativos, `c.comp` normalmente fica em 0 (exceto o Insertion Sort usado dentro dos baldes do Bucket Sort). Isso é esperado e **deve ser discutido**.
5. **Integridade acadêmica:** o código e o texto devem ser de autoria do grupo. Consultar livros e materiais é permitido e deve ser **citado**. Trabalhos com trechos idênticos entre grupos, ou gerados integralmente por ferramentas de IA sem compreensão demonstrada, receberão nota zero. O professor poderá solicitar uma **arguição oral** sobre qualquer parte do trabalho.

<a id="dicas"></a>
## 3. Dicas práticas para iniciantes ⚠️

### 3.1 Seeds: tornando a aleatoriedade reprodutível
Um gerador pseudoaleatório produz sempre a **mesma sequência** a partir da mesma *seed* (semente). Isso permite que você (e o professor) reproduza exatamente os mesmos vetores.

```python
import random
rng = random.Random(42)        # gerador PRÓPRIO, com seed 42
v = [rng.randint(0, 100) for _ in range(10)]
```
- Prefira `random.Random(seed)` a `random.seed(seed)`: um gerador local não é afetado por outras partes do código (ex.: o pivô aleatório do Quicksort).
- **Todos os algoritmos devem receber exatamente o mesmo vetor** para uma dada combinação (tipo, n, seed). O código de apoio já faz isso — ele gera o vetor uma vez e entrega uma **cópia** para cada algoritmo.
- Use várias seeds (usaremos 5) para estimar a **variabilidade**: um único vetor aleatório pode ser "sortudo" ou "azarado".

### 3.2 Medindo tempo corretamente
- Use `time.perf_counter()` (relógio de alta resolução). **Não** use `time.time()` nem `datetime`.
- Meça **apenas** a chamada do algoritmo — não inclua a geração do vetor, a cópia nem a verificação.
- Desative o *garbage collector* durante a medição (`gc.disable()` / `gc.enable()`) para reduzir ruído.
- Feche outros programas pesados e **não use o computador** durante o modo completo.
- Tempos abaixo de ~1 ms são muito ruidosos. Para *n* pequeno, a média de várias seeds ajuda.
- A instrumentação (contadores) acrescenta um custo aproximadamente constante por operação. Isso **altera as constantes, mas não a ordem de crescimento** — e todos os algoritmos pagam esse custo. Mencione isso na metodologia.

### 3.3 Limite de recursão
O Python limita a profundidade de recursão (padrão ≈ 1000). O Quicksort com pivô no primeiro elemento em um vetor **já ordenado** faz recursão de profundidade *n* e gera `RecursionError`.

Estratégias (use as duas primeiras):
1. **Recurse primeiro na menor partição e trate a maior com um laço** (eliminação de recursão de cauda). Isso garante profundidade $O(\log n)$ mesmo no pior caso.
2. Use **pivô aleatório** ou **mediana de três** na versão principal.
3. Se ainda assim precisar, aumente o limite com moderação: `sys.setrecursionlimit(10**5)`. Valores muito altos podem **derrubar o kernel** (estouro da pilha do sistema operacional). Se o kernel morrer, reinicie-o e reduza *n*.

O Merge Sort tem profundidade $\lceil \log_2 n \rceil$ (≈ 20 para $n = 10^6$), portanto não tem esse problema.

### 3.4 Outras armadilhas comuns
- **Aliasing:** `w = v` **não** copia a lista. Use `w = v[:]` ou `list(v)`.
- **Slices criam cópias:** `v[inicio:fim]` custa $O(fim - inicio)$ de tempo e memória. No Merge Sort isso é aceitável se contabilizado; no Quicksort, trabalhe com índices `(inicio, fim)`.
- **Off-by-one:** decida se `fim` é inclusivo ou exclusivo e seja consistente. Escreva isso num comentário.
- **Rode sempre do início** (`Kernel → Restart & Run All`) antes de entregar, para garantir que o notebook não depende de células executadas fora de ordem.
- **Salve os resultados** dos experimentos em CSV (o código de apoio faz isso). Assim você não precisa rodar tudo de novo se o kernel reiniciar.

<a id="apoio"></a>
## 4. Código de apoio 🧰

Execute as células abaixo na ordem. Leia os comentários: eles explicam o que cada função faz e como você deve usá-la.

In [1]:
# 🧰 Importações e configuração geral
import sys, gc, math, time, random, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.setrecursionlimit(10_000)   # moderado; veja a Dica 3.3
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
print("Python", sys.version.split()[0])

Python 3.14.4


### 4.1 O contador de operações
Todo algoritmo recebe um objeto `Contador`. Você incrementa `c.comp` a cada comparação entre elementos e `c.trocas` a cada movimentação. As funções auxiliares `menor`, `trocar` e `escrever` fazem a contagem por você — **use-as**, pois reduzem esquecimentos.

In [ ]:
# 🧰 Instrumentação
class Contador:
    # Guarda o número de comparações (comp) e de movimentações (trocas)
    __slots__ = ("comp", "trocas")
    def __init__(self):
        self.comp = 0
        self.trocas = 0
    def __repr__(self):
        return f"Contador(comp={self.comp}, trocas={self.trocas})"


def menor(a, b, c):
    # Retorna a < b e conta UMA comparação.
    c.comp += 1
    return a < b

def menor_ou_igual(a, b, c):
    # Retorna a <= b e conta UMA comparação (útil para manter estabilidade).
    c.comp += 1
    return a <= b

def trocar(v, i, j, c):
    # Troca v[i] e v[j] e conta UMA troca.
    v[i], v[j] = v[j], v[i]
    c.trocas += 1

def escrever(v, i, valor, c):
    # Escreve valor em v[i] e conta UMA movimentação.
    v[i] = valor
    c.trocas += 1

**Exemplo de uso** (não é um dos algoritmos do TP): encontrar o índice do menor elemento de um trecho do vetor. Observe **onde** a contagem acontece.

In [ ]:
# 🧰 Exemplo didático de instrumentação
def indice_do_minimo(v, inicio, c):
    idx = inicio
    for k in range(inicio + 1, len(v)):
        if menor(v[k], v[idx], c):   # comparação entre ELEMENTOS -> conta
            idx = k                   # atribuição de ÍNDICE -> não conta
    return idx

c = Contador()
print(indice_do_minimo([7, 3, 9, 1, 5], 0, c), c)   # esperado: 3, comp=4

### 4.2 Geradores de entrada
- `gerar_inteiros(tipo, n, seed, k)` produz inteiros em $[0, k)$ nos tipos: `"aleatoria"`, `"ordenada"`, `"inversa"` e `"quase_ordenada"` (vetor ordenado com ~1% de pares trocados aleatoriamente).
- `gerar_floats(n, seed, distribuicao)` produz reais em $[0, 1)$ para o Bucket Sort, com distribuição `"uniforme"` ou `"concentrada"` (90% dos valores em $[0, 0{,}1)$).

> Os tipos `"ordenada"` e `"inversa"` são gerados sem chamar `sorted()`: construímos a sequência crescente diretamente.

In [ ]:
# 🧰 Geradores de entrada
TIPOS = ["aleatoria", "ordenada", "inversa", "quase_ordenada"]

def gerar_inteiros(tipo, n, seed, k=None):
    # Gera uma lista de n inteiros em [0, k). Se k for None, usa k = n.
    rng = random.Random(seed)
    if k is None:
        k = max(n, 1)
    if tipo == "aleatoria":
        return [rng.randrange(k) for _ in range(n)]
    # sequência não decrescente espalhada em [0, k), sem usar sorted()
    crescente = [(i * k) // max(n, 1) for i in range(n)]
    if tipo == "ordenada":
        return crescente
    if tipo == "inversa":
        return crescente[::-1]
    if tipo == "quase_ordenada":
        v = crescente
        for _ in range(max(1, n // 100)):          # ~1% de trocas aleatórias
            i, j = rng.randrange(n), rng.randrange(n)
            v[i], v[j] = v[j], v[i]
        return v
    raise ValueError(f"tipo desconhecido: {tipo}")


def gerar_floats(n, seed, distribuicao="uniforme"):
    # Gera n reais em [0, 1) para o Bucket Sort.
    rng = random.Random(seed)
    if distribuicao == "uniforme":
        return [rng.random() for _ in range(n)]
    if distribuicao == "concentrada":
        return [rng.random() * 0.1 if rng.random() < 0.9 else rng.random()
                for _ in range(n)]
    raise ValueError(f"distribuição desconhecida: {distribuicao}")


for t in TIPOS:
    print(f"{t:>15}:", gerar_inteiros(t, 12, seed=1, k=50))

### 4.3 Verificação de ordenação
`esta_ordenado(v)` verifica em $O(n)$ se a lista está em ordem não decrescente. `mesma_multiset(a, b)` verifica se as duas listas têm os mesmos elementos (nenhum elemento foi perdido ou duplicado).

In [2]:
# 🧰 Verificação
from collections import Counter as _Multiset

def esta_ordenado(v):
    return all(v[i] <= v[i + 1] for i in range(len(v) - 1))

def mesma_multiset(a, b):
    return _Multiset(a) == _Multiset(b)

<a id="parteA"></a>
# Parte A — Algoritmos quadráticos

📘 **Ideia comum:** os três algoritmos desta parte mantêm uma **região ordenada** que cresce uma posição por iteração do laço externo. Como cada iteração pode custar $\Theta(n)$ e há $n$ iterações, o custo típico é $\Theta(n^2)$. Apesar disso, eles são simples, *in-place* ($O(1)$ de memória extra) e — como você verá — podem ser **mais rápidos que os algoritmos $O(n \log n)$ para $n$ pequeno**.

Um conceito que aparece nesta parte é o de **inversão**: um par de índices $(i, j)$ com $i < j$ e $v[i] > v[j]$. Um vetor ordenado tem 0 inversões; um vetor inversamente ordenado tem $\binom{n}{2} = \frac{n(n-1)}{2}$ inversões; um vetor aleatório tem, em média, $\frac{n(n-1)}{4}$.

## A.1 Bubble Sort 📘

**Ideia.** Percorre o vetor comparando pares adjacentes e trocando-os quando estão fora de ordem. Ao final da passada $i$, o $i$-ésimo maior elemento "borbulhou" até sua posição final.

**Invariante do laço externo.** Após $i$ passadas, as $i$ últimas posições contêm os $i$ maiores elementos, em ordem.

**Otimização obrigatória (parada antecipada).** Se uma passada inteira não fizer nenhuma troca, o vetor já está ordenado e o algoritmo pode parar.

**Pseudocódigo**
```
BUBBLE-SORT(v)
  n ← |v|
  para i ← 0 até n-2
      trocou ← falso
      para j ← 0 até n-2-i
          se v[j] > v[j+1]
              troque v[j] e v[j+1]
              trocou ← verdadeiro
      se não trocou: pare
```

**Pontos para refletir.** Quantas trocas o Bubble Sort faz em relação ao número de inversões do vetor? O que acontece com a parada antecipada num vetor já ordenado?

In [ ]:
# 🛠️ Bubble Sort
def bubble_sort(v, c):
    n = len(v)
    for i in range(n):
        trocou = False
        for j in range(n - i - 1):
            if menor(v[j + 1], v[j], c):
                trocar(v, j, j + 1, c)
                trocou = True
        if not trocou:
            break
    return v

## A.2 Selection Sort 📘

**Ideia.** Na iteração $i$, **seleciona** o menor elemento de $v[i..n-1]$ e o coloca na posição $i$ com uma única troca.

**Invariante.** Antes da iteração $i$, $v[0..i-1]$ contém os $i$ menores elementos, em ordem.

**Pseudocódigo**
```
SELECTION-SORT(v)
  n ← |v|
  para i ← 0 até n-2
      m ← i
      para j ← i+1 até n-1
          se v[j] < v[m]: m ← j
      se m ≠ i: troque v[i] e v[m]
```

**Pontos para refletir.** O número de comparações depende da entrada? E o número de trocas? O Selection Sort é **estável**? (Dica: tente o vetor `[(2,'a'), (2,'b'), (1,'c')]` comparando só a primeira coordenada.)

In [ ]:
# 🛠️ Selection Sort
def selection_sort(v, c):
    n = len(v)
    for i in range(n - 1):
        m = i
        for j in range(i + 1, n):
            if menor(v[j], v[m], c):
                m = j
        if m != i:
            trocar(v, i, m, c)
    return v

## A.3 Insertion Sort 📘

**Ideia.** Como organizar cartas na mão: pega o próximo elemento (a *chave*) e o **insere** na posição correta da região já ordenada, deslocando para a direita os elementos maiores.

**Invariante.** Antes da iteração $i$, $v[0..i-1]$ contém os elementos originais dessas posições, em ordem.

**Pseudocódigo**
```
INSERTION-SORT(v)
  para i ← 1 até n-1
      chave ← v[i]
      j ← i - 1
      enquanto j ≥ 0 e v[j] > chave
          v[j+1] ← v[j]          ▹ deslocamento: conta 1 movimentação
          j ← j - 1
      v[j+1] ← chave             ▹ conta 1 movimentação
```

⚠️ **Contagem cuidadosa de comparações.** Na condição `j ≥ 0 e v[j] > chave`, a comparação `v[j] > chave` só acontece quando `j ≥ 0`. Conte-a exatamente quando ela for avaliada.

**Pontos para refletir.** Relacione o número de deslocamentos com o número de inversões. Por que o Insertion Sort é tão bom em vetores **quase ordenados**? Essa propriedade será usada no Shellsort e no Bucket Sort.

> 💡 Implemente também `insertion_sort_intervalo(v, inicio, fim, c)`, que ordena apenas `v[inicio..fim]` (inclusivo). Ela é útil para a questão de discussão G.1 (algoritmos híbridos) e pode ser reaproveitada no Bucket Sort.

In [ ]:
# 🛠️ Insertion Sort
def insertion_sort_intervalo(v, inicio, fim, c):
    # fim é inclusivo
    for i in range(inicio + 1, fim + 1):
        chave = v[i]
        j = i
        while j > inicio and menor(chave, v[j - 1], c):
            escrever(v, j, v[j - 1], c)
            j -= 1
        if j != i:
            escrever(v, j, chave, c)
    return v

def insertion_sort(v, c):
    return insertion_sort_intervalo(v, 0, len(v) - 1, c)

## A.4 Análise teórica dos quadráticos ✍️

| Algoritmo | Melhor caso | Caso médio | Pior caso | Entrada do melhor / pior caso | Espaço extra | Estável? | Comparações exatas no pior caso | Trocas no pior caso |
|---|---|---|---|---|---|---|---|---|
| **Bubble Sort** | $\Theta(n)$ | $\Theta(n^2)$ | $\Theta(n^2)$ | **Melhor:** Vetor já ordenado<br>**Pior:** Vetor inversamente ordenado | $O(1)$ | Sim | $\frac{n(n-1)}{2}$ | $\frac{n(n-1)}{2}$ |
| **Selection Sort** | $\Theta(n^2)$ | $\Theta(n^2)$ | $\Theta(n^2)$ | **Melhor:** Qualquer entrada<br>**Pior:** Qualquer entrada | $O(1)$ | Não | $\frac{n(n-1)}{2}$ | $n - 1$ |
| **Insertion Sort** | $\Theta(n)$ | $\Theta(n^2)$ | $\Theta(n^2)$ | **Melhor:** Vetor já ordenado<br>**Pior:** Vetor inversamente ordenado | $O(1)$ | Sim | $\frac{n(n-1)}{2}$ | $\frac{n(n-1)}{2}$ |

---

### Justificativas:

#### 1. Bubble Sort (com flag de otimização para parada antecipada)

* **Análise de Comparações e Trocas (Pior Caso):**
  O pior caso ocorre quando a entrada está **inversamente ordenada**. O laço externo executa $n-1$ passagens. Na $i$-ésima passagem ($i = 1, 2, \dots, n-1$), são feitas $n-i$ comparações e $n-i$ trocas para flutuar o maior elemento até o final:
  $$\text{Comparações}(n) = \sum_{i=1}^{n-1} (n - i) = \sum_{j=1}^{n-1} j = \frac{n(n-1)}{2} = \frac{n^2 - n}{2} = \Theta(n^2)$$
  Como cada comparação resulta em uma troca no pior caso:
  $$\text{Trocas}(n) = \sum_{i=1}^{n-1} (n - i) = \frac{n(n-1)}{2} = \Theta(n^2)$$

* **Melhor Caso:**
  Ocorre quando o vetor já está **ordenado**. A flag de verificação detecta que nenhuma troca foi realizada no primeiro passo do laço externo ($i=1$) e interrompe o algoritmo imediatamente.
  $$\text{Comparações}(n) = n - 1 = \Theta(n), \quad \text{Trocas}(n) = 0$$

* **Caso Médio:**
  Para uma permutação aleatória de $n$ elementos, o número esperado de inversões é $\frac{n(n-1)}{4}$. O número esperado de comparações e trocas permanece limitado por $\Theta(n^2)$.

* **Espaço Extra e Estabilidade:**
  * **Espaço:** $O(1)$ (*in-place*), utilizando apenas variáveis auxiliares para a troca e controle de laço.
  * **Estável:** Elementos de valores iguais nunca trocam de posição entre si, pois a condição de troca é estritamente de maior que (`A[j] > A[j+1]`).

---

#### 2. Selection Sort

* **Análise de Comparações e Trocas (Pior Caso e Geral):**
  O Selection Sort percorre o vetor para encontrar o menor elemento da sublista não ordenada e o troca com o elemento da posição atual $i$.
  O número de comparações é rigorosamente **fixo** para qualquer tipo de entrada (ordenada, inversa ou aleatória):
  $$\text{Comparações}(n) = \sum_{i=1}^{n-1} (n - i) = \frac{n(n-1)}{2} = \Theta(n^2)$$
  Em cada uma das $n-1$ iterações do laço externo, realiza-se no máximo uma troca de posição (swap):
  $$\text{Trocas}(n) \le n - 1 = \Theta(n)$$

* **Melhor, Médio e Pior Caso:**
  Como a busca pelo mínimo exige varrer obrigatoriamente todo o restante da sublista não ordenada a cada iteração, o número de comparações não varia com o arranjo inicial dos dados. Portanto, **melhor, médio e pior caso coincidem em $\Theta(n^2)$**.

* **Espaço Extra e Estabilidade:**
  * **Espaço:** $O(1)$ (*in-place*).
  * **Instável:** Ao trocar o menor elemento encontrado com o elemento da posição atual $i$, um elemento de valor igual que estivesse posicionado entre $i$ e a posição do menor pode ser deslocado para trás, alterando a ordem relativa original entre eles.

---

#### 3. Insertion Sort

* **Análise de Comparações e Deslocamentos (Pior Caso):**
  O pior caso ocorre quando o vetor está **inversamente ordenado**. Para cada elemento na posição $i$ ($i = 1, 2, \dots, n-1$), o laço interno precisa deslocar todos os $i$ elementos anteriores já ordenados para abrir espaço:
  $$\text{Comparações}(n) = \sum_{i=1}^{n-1} i = \frac{n(n-1)}{2} = \Theta(n^2)$$
  Cada deslocamento no laço interno equivale a uma movimentação/troca de elemento:
  $$\text{Trocas/Atribuições}(n) = \sum_{i=1}^{n-1} i = \frac{n(n-1)}{2} = \Theta(n^2)$$

* **Melhor Caso:**
  Ocorre quando a entrada está **totalmente ordenada**. Ao testar o elemento $A[i]$ contra $A[i-1]$, o laço interno encerra imediatamente na primeira comparação (pois $A[i] \ge A[i-1]$), sem realizar deslocamentos:
  $$\text{Comparações}(n) = \sum_{i=1}^{n-1} 1 = n - 1 = \Theta(n), \quad \text{Trocas/Deslocamentos}(n) = 0$$

* **Caso Médio:**
  Em uma entrada aleatória, cada elemento insere-se em média na metade da sublista já ordenada ($\frac{i}{2}$ comparações no passo $i$):
  $$\text{Comparações}(n) \approx \sum_{i=1}^{n-1} \frac{i}{2} = \frac{n(n-1)}{4} = \Theta(n^2)$$

* **Espaço Extra e Estabilidade:**
  * **Espaço:** $O(1)$ (*in-place*).
  * **Estável:** O laço interno de inserção interrompe o deslocamento assim que encontra um elemento menor ou igual (`A[j] <= pivo`), preservando a ordem relativa de elementos idênticos.

<a id="parteB"></a>
# Parte B — Algoritmos subquadráticos

📘 **Relembrando o Teorema Mestre.** Seja $T(n) = a\,T(n/b) + f(n)$, com $a \ge 1$, $b > 1$. Compare $f(n)$ com $n^{\log_b a}$:

1. Se $f(n) = O(n^{\log_b a - \varepsilon})$ para algum $\varepsilon > 0$, então $T(n) = \Theta(n^{\log_b a})$.
2. Se $f(n) = \Theta(n^{\log_b a})$, então $T(n) = \Theta(n^{\log_b a} \log n)$.
3. Se $f(n) = \Omega(n^{\log_b a + \varepsilon})$ para algum $\varepsilon > 0$ **e** $a f(n/b) \le c f(n)$ para algum $c < 1$ e $n$ suficientemente grande, então $T(n) = \Theta(f(n))$.

⚠️ O Teorema Mestre **não** se aplica a recorrências que não têm a forma $a\,T(n/b) + f(n)$ — por exemplo, $T(n) = T(n-1) + \Theta(n)$ (pior caso do Quicksort). Nesses casos use **substituição** ou **árvore de recursão**.

## B.1 Shellsort 📘

**Ideia.** Generaliza o Insertion Sort: em vez de comparar só vizinhos, ordena por inserção os elementos que estão a distância $h$ (*gap*) uns dos outros, para uma **sequência decrescente de gaps** terminando em $h = 1$. Gaps grandes movem elementos para longe rapidamente; quando chega em $h = 1$, o vetor já está "quase ordenado" e o Insertion Sort final é barato.

Um vetor em que $v[i] \le v[i+h]$ para todo $i$ é dito **$h$-ordenado**. Um resultado importante: *h-ordenar um vetor que já é k-ordenado mantém a k-ordenação.*

**Pseudocódigo**
```
SHELLSORT(v, gaps)                 ▹ gaps em ordem DECRESCENTE, último = 1
  para cada h em gaps
      para i ← h até n-1
          chave ← v[i]
          j ← i
          enquanto j ≥ h e v[j-h] > chave
              v[j] ← v[j-h]
              j ← j - h
          v[j] ← chave
```

**Sequências de gaps que você deve implementar e comparar:**

| Nome | Definição | Primeiros termos | Pior caso conhecido |
|---|---|---|---|
| **Shell (1959)** | $\lfloor n/2 \rfloor, \lfloor n/4 \rfloor, \dots, 1$ | depende de $n$ | $\Theta(n^2)$ |
| **Knuth (1973)** | $h_1 = 1,\ h_{k+1} = 3h_k + 1$, enquanto $h < n/3$ | 1, 4, 13, 40, 121, 364, … | $O(n^{3/2})$ |
| **Ciura (2001)** | empírica; estenda com $h_{k+1} = \lfloor 2{,}25\,h_k \rfloor$ | 1, 4, 10, 23, 57, 132, 301, 701, 1750, … | desconhecido (excelente na prática) |

> A análise do caso médio do Shellsort é um **problema em aberto** para a maioria das sequências. Não se espera que você a demonstre; espera-se que você **cite os resultados conhecidos** e **estime empiricamente** o expoente $\alpha$ em $T(n) \approx c\,n^\alpha$ (a função `estimar_expoente` da Parte F faz isso).

In [ ]:
# 🛠️ Sequências de gaps (ordem decrescente, terminando em 1)
def gaps_shell(n):
    gaps, g = [], n // 2
    while g > 0:
        gaps.append(g)
        g //= 2
    return gaps

def gaps_knuth(n):
    gaps = [1]
    while 3 * gaps[-1] + 1 < n / 3:
        gaps.append(3 * gaps[-1] + 1)
    return [g for g in reversed(gaps) if g < n]

def gaps_ciura(n):
    base = [1, 4, 10, 23, 57, 132, 301, 701, 1750]
    while base[-1] * 2.25 < n:
        base.append(int(base[-1] * 2.25))
    return [g for g in reversed(base) if g < n]

# 🛠️ Shellsort
def shellsort(v, c, gaps_fn=None):
    if gaps_fn is None:
        gaps_fn = gaps_ciura
    n = len(v)
    for h in gaps_fn(n):
        # inserção com salto h
        for i in range(h, n):
            chave = v[i]
            j = i
            while j >= h and menor(chave, v[j - h], c):
                escrever(v, j, v[j - h], c)
                j -= h
            if j != i:
                escrever(v, j, chave, c)
    return v

def shellsort_shell(v, c): return shellsort(v, c, gaps_shell)
def shellsort_knuth(v, c): return shellsort(v, c, gaps_knuth)
def shellsort_ciura(v, c): return shellsort(v, c, gaps_ciura)

## B.2 Merge Sort 📘

**Ideia (divisão e conquista).**
1. **Dividir:** parte o vetor ao meio.
2. **Conquistar:** ordena recursivamente cada metade.
3. **Combinar:** **intercala** (*merge*) as duas metades ordenadas em tempo linear.

**Pseudocódigo** (índices com `fim` exclusivo)
```
MERGE-SORT(v, ini, fim, aux)
  se fim - ini ≤ 1: retorne
  meio ← ⌊(ini + fim) / 2⌋
  MERGE-SORT(v, ini, meio, aux)
  MERGE-SORT(v, meio, fim, aux)
  MERGE(v, ini, meio, fim, aux)

MERGE(v, ini, meio, fim, aux)
  copie v[ini..fim) para aux[ini..fim)
  i ← ini;  j ← meio
  para k ← ini até fim-1
      se i ≥ meio:           v[k] ← aux[j]; j ← j+1
      senão se j ≥ fim:      v[k] ← aux[i]; i ← i+1
      senão se aux[j] < aux[i]:  v[k] ← aux[j]; j ← j+1     ▹ compara elementos
      senão:                 v[k] ← aux[i]; i ← i+1         ▹ empate vai para a esquerda → ESTÁVEL
```

**Dicas de implementação.**
- Aloque o vetor auxiliar **uma única vez** (tamanho $n$) e passe-o para as chamadas recursivas. Alocar dentro do `MERGE` a cada chamada funciona, mas é mais lento.
- Conte como movimentação cada escrita `v[k] ← ...`. As cópias para `aux` também podem ser contadas — **decida e documente** na metodologia.

**Recorrência.** $T(n) = 2\,T(n/2) + \Theta(n)$, $T(1) = \Theta(1)$.

In [ ]:
# 🛠️ Merge Sort
def merge_sort(v, c):
    n = len(v)
    if n < 2:
        return v
    aux = [None] * n
    _merge_sort_rec(v, aux, 0, n - 1, c)
    return v

def _merge_sort_rec(v, aux, lo, hi, c):
    # lo e hi inclusivos
    if lo >= hi:
        return
    mid = (lo + hi) // 2
    _merge_sort_rec(v, aux, lo, mid, c)
    _merge_sort_rec(v, aux, mid + 1, hi, c)
    _intercalar(v, aux, lo, mid, hi, c)

def _intercalar(v, aux, lo, mid, hi, c):
    aux[lo:hi + 1] = v[lo:hi + 1]   # cópia para aux não é contada
    i, j = lo, mid + 1
    for k in range(lo, hi + 1):
        if i > mid:
            escrever(v, k, aux[j], c); j += 1
        elif j > hi:
            escrever(v, k, aux[i], c); i += 1
        elif menor(aux[j], aux[i], c):   # estrito: mantém estabilidade
            escrever(v, k, aux[j], c); j += 1
        else:
            escrever(v, k, aux[i], c); i += 1

## B.3 Quicksort 📘

**Ideia (divisão e conquista).**
1. **Dividir:** escolhe um **pivô** e **particiona** o vetor de modo que os elementos menores fiquem à esquerda e os maiores à direita.
2. **Conquistar:** ordena recursivamente as duas partes.
3. **Combinar:** nada a fazer — o pivô já está na posição final.

**Partição de Lomuto** (pivô na última posição, índices inclusivos)
```
PARTICIONA-LOMUTO(v, ini, fim)
  p ← v[fim]
  i ← ini - 1
  para j ← ini até fim-1
      se v[j] ≤ p
          i ← i + 1
          troque v[i] e v[j]
  troque v[i+1] e v[fim]
  retorne i + 1
```
Para usar outra estratégia de pivô, basta **trocar o pivô escolhido com `v[fim]`** antes de chamar a partição.

**Estratégias de pivô a implementar:**
| Estratégia | Como escolher o índice do pivô |
|---|---|
| `"ultimo"` | `fim` |
| `"aleatorio"` | índice sorteado uniformemente em `[ini, fim]` com um `random.Random` próprio |
| `"mediana3"` | o índice do valor mediano entre `v[ini]`, `v[meio]`, `v[fim]` (conte as comparações!) |

**Controle da profundidade de recursão (obrigatório).** Implemente o laço abaixo, que chama a recursão apenas na **menor** parte:
```
QUICKSORT(v, ini, fim)
  enquanto ini < fim
      q ← PARTICIONA(v, ini, fim)
      se q - ini < fim - q
          QUICKSORT(v, ini, q-1);  ini ← q + 1
      senão
          QUICKSORT(v, q+1, fim);  fim ← q - 1
```
Isso garante profundidade $O(\log n)$, mas **não** muda o tempo de pior caso.

⚠️ **Elementos repetidos.** Com Lomuto, um vetor com muitos valores iguais (ex.: $k$ pequeno) leva ao pior caso. Isso **não** precisa ser corrigido, mas deve ser mencionado se aparecer nos seus experimentos. (Opcional, para quem quiser ir além: partição de 3 vias de Dijkstra.)

**Recorrências.**
- Melhor caso (partições equilibradas): $T(n) = 2\,T(n/2) + \Theta(n)$.
- Pior caso (uma parte vazia): $T(n) = T(n-1) + \Theta(n)$.
- Caso médio (pivô uniformemente aleatório): $T(n) = \Theta(n) + \frac{1}{n}\sum_{q=0}^{n-1}\big[T(q) + T(n-1-q)\big]$.

In [ ]:
# 🛠️ Quicksort
def quicksort(v, c, pivo="aleatorio", seed=0):
    if pivo not in ("ultimo", "aleatorio", "mediana3"):
        raise ValueError(f"Pivô inválido: {pivo}")
    rng = random.Random(seed)
    _quicksort_rec(v, 0, len(v) - 1, pivo, rng, c)
    return v

def _quicksort_rec(v, lo, hi, pivo, rng, c):
    # lo e hi inclusivos; recursão só na menor parte
    while lo < hi:
        p = _particionar(v, lo, hi, pivo, rng, c)
        if p - lo < hi - p:
            _quicksort_rec(v, lo, p - 1, pivo, rng, c)
            lo = p + 1
        else:
            _quicksort_rec(v, p + 1, hi, pivo, rng, c)
            hi = p - 1

def _escolher_pivo(v, lo, hi, pivo, rng, c):
    # move o pivô escolhido para v[hi]
    if pivo == "aleatorio":
        k = rng.randint(lo, hi)
    elif pivo == "mediana3":
        a, b, d = lo, (lo + hi) // 2, hi
        if menor(v[a], v[b], c):
            if menor(v[b], v[d], c):
                k = b
            elif menor(v[a], v[d], c):
                k = d
            else:
                k = a
        else:
            if menor(v[a], v[d], c):
                k = a
            elif menor(v[b], v[d], c):
                k = d
            else:
                k = b
    else:
        k = hi

    if k != hi:
        trocar(v, k, hi, c)

def _particionar(v, lo, hi, pivo, rng, c):
    # Lomuto: elementos <= pivô vão para a esquerda
    _escolher_pivo(v, lo, hi, pivo, rng, c)
    p = v[hi]
    i = lo
    for j in range(lo, hi):
        if menor_ou_igual(v[j], p, c):
            if i != j:
                trocar(v, i, j, c)
            i += 1
    if i != hi:
        trocar(v, i, hi, c)
    return i

def quicksort_aleatorio(v, c): return quicksort(v, c, "aleatorio")
def quicksort_ultimo(v, c):    return quicksort(v, c, "ultimo")
def quicksort_mediana3(v, c):  return quicksort(v, c, "mediana3")

## B.4 Análise teórica dos subquadráticos ✍️

### B.4.1 Merge Sort

#### 1. Aplicação do Teorema Mestre a $T(n) = 2T(n/2) + \Theta(n)$
* **Identificação dos parâmetros:** $a = 2$, $b = 2$, $f(n) = \Theta(n)$.
* **Cálculo de $n^{\log_b a}$:** $n^{\log_2 2} = n^1 = n$.
* **Caso do Teorema Mestre:** Como $f(n) = \Theta(n^{\log_b a}) = \Theta(n)$, aplica-se o **Caso 2 do Teorema Mestre**.
* **Conclusão:** $T(n) = \Theta(n^{\log_b a} \log n) = \Theta(n \log n)$.

#### 2. Por que os casos coincidem?
A estrutura de divisão e intercalação do Merge Sort é estritamente **determinística** e independente dos valores dos elementos no vetor. O vetor é incondicionalmente dividido ao meio até atingir subvetores unitários e, em seguida, intercala-se $n$ elementos por nível de árvore (fazendo até $n-1$ comparações e exatamente $n$ movimentações). Como a altura da árvore é sempre $\lceil \log_2 n \rceil$ e o trabalho por nível é $\Theta(n)$, o **melhor, médio e pior caso coincidem em $\Theta(n \log n)$**.

#### 3. Espaço extra e estabilidade
* **Espaço extra:** $O(n)$, pois a etapa de intercalação (*merge*) necessita de um vetor auxiliar para armazenar temporariamente os elementos ordenados antes de copiá-los de volta para o vetor principal.
* **Estabilidade:** É **estável**. Durante a intercalação, quando há empate entre elementos do subvetor da esquerda e da direita ($A[i] \le B[j]$), o elemento do subvetor da esquerda é prioritariamente inserido no vetor final, preservando a ordem relativa original.

---

### B.4.2 Quicksort

#### 1. Melhor caso
Ocorre quando o pivô sempre divide o vetor em duas metades exatamente iguais de tamanho $n/2$.
* Recorrência: $T(n) = 2T(n/2) + \Theta(n)$.
* Pelo **Caso 2 do Teorema Mestre** ($a=2, b=2, f(n)=\Theta(n)$): $T(n) = \Theta(n \log n)$.

#### 2. Pior caso
* **Aplicação do Teorema Mestre:** **Não se aplica** a $T(n) = T(n-1) + \Theta(n)$, pois a relação não está na forma $T(n) = a T(n/b) + f(n)$ (a divisão não é proporcional por uma constante $b > 1$, mas sim uma subtração).
* **Resolução por expansão:**
  $$T(n) = T(n-1) + c n$$
  $$T(n) = T(n-2) + c(n-1) + c n$$
  $$T(n) = T(1) + c \sum_{i=2}^{n} i = T(1) + c \left( \frac{n(n+1)}{2} - 1 \right) = \Theta(n^2)$$
* **Entradas que levam ao pior caso:**
  * **Com estratégia `"ultimo"`:** Vetores já **ordenados** ou **inversamente ordenados**. Em cada passo, o pivô escolhido é o mínimo ou o máximo do subvetor, gerando divisões de tamanho $0$ e $n-1$.
  * **Com estratégia `"aleatorio"`:** Nenhuma entrada específica garante o pior caso deterministicamente. O pior caso só ocorre se o gerador aleatório sortear sistematicamente o maior ou menor elemento em *todas* as chamadas recursivas (probabilidade $\frac{2^n}{n!}$).

#### 3. Partição desbalanceada fixa ($1 : 9$)
Recorrência: $T(n) = T(n/10) + T(9n/10) + c n$.

* **Aplicação do Teorema Mestre:** **Não se aplica** diretamente, pois as subchamadas possuem tamanhos diferentes ($n/10$ e $9n/10$).
* **Árvore de recursão:**
  * O trabalho no nó raiz é $c n$.
  * No nível 1, os filhos custam $c(n/10) + c(9n/10) = c n$.
  * Em qualquer nível completo $k$, a soma das entradas resulta em $c n$.
  * O caminho mais curto atinge as folhas na profundidade $\log_{10} n$.
  * O caminho mais longo atinge as folhas na profundidade $\log_{10/9} n = \Theta(\log n)$.
  * Somando a cota superior de todos os níveis: $T(n) \le c n \cdot \log_{10/9} n = \Theta(n \log n)$.
  * **Conclusão:** Mesmo um desbalanceamento constante e severo ($90\% / 10\%$) resulta em tempo $\Theta(n \log n)$.

#### 4. Caso médio
Conforme demonstrado em Cormen et al. (Cap. 7), sob a hipótese de que todas as permutações da entrada são igualmente prováveis, o tempo esperado do Quicksort é $\Theta(n \log n)$.
* **Número esperado de comparações:** $C(n) \approx 2n \ln n \approx 1{,}39\,n \log_2 n$.
* **Ideia da prova:** Define-se variáveis indicadoras $X_{ij}$ para a comparação entre o $i$-ésimo e o $j$-ésimo menor elemento. A probabilidade de $z_i$ e $z_j$ serem comparados é a chance de um deles ser escolhido como pivô antes de qualquer outro elemento no intervalo entre eles, resultando em $P(z_i \text{ é comparado a } z_j) = \frac{2}{j - i + 1}$. Somando essa probabilidade sobre todos os pares $1 \le i < j \le n$, obtém-se a série harmônica que resulta em $2n \ln n$.

#### 5. Espaço (Pilha de Recursão)
* **Sem otimização (recursão simples):** No pior caso $\Theta(n^2)$, a profundidade da pilha atinge $O(n)$, podendo causar estouro de pilha (*stack overflow*).
* **Com otimização da menor parte (Tail Recursion Elimination):** Garantindo que a chamada recursiva seja feita primeiro para o menor subvetor e usando um laço para o maior, a profundidade máxima da pilha fica limitada a $O(\log n)$ no pior caso.

---

### B.4.3 Shellsort

#### Limites de pior caso conhecidos
1. **Shell original ($\lfloor n/2^k \rfloor$):** $\Theta(n^2)$
2. **Knuth ($h_k = 3h_{k-1} + 1$):** $O(n^{3/2}) = O(n^{1{,}5})$
3. **Ciura ($[1, 4, 10, 23, 57, 132, 301, 701, \dots]$):** nenhum limite de pior caso demonstrado; a sequência foi obtida experimentalmente, para o caso médio (CIURA, 2001)

#### Por que a sequência original de Shell pode ser ruim?
Na sequência de Shell original ($n/2, n/4, n/8, \dots, 1$), quando $n$ é potência de 2, todos os *gaps* intermediários também são potências de 2. Isso gera o problema dos **fatores comuns**: elementos localizados em posições pares só são comparados com elementos em posições pares até que o gap final $h = 1$ seja atingido. 

Se uma entrada colocar os maiores elementos nas posições ímpares e os menores nas posições pares, as passagens com $h = 2^k$ não realizarão nenhuma ordenação útil entre pares e ímpares. Todo o trabalho pesado de ordenação fica acumulado na última etapa ($h = 1$), reduzindo o algoritmo ao funcionamento do Insertion Sort convencional e elevando o tempo para $\Theta(n^2)$.

---

### B.4.4 Tabela-resumo

| Algoritmo | Melhor | Médio | Pior | Espaço extra | Estável? | In-place? |
|---|---|---|---|---|---|---|
| **Shellsort (Shell)** | $O(n \log n)$ | em aberto | $\Theta(n^2)$ | $O(1)$ | Não | Sim |
| **Shellsort (Knuth)** | $O(n \log n)$ | em aberto (empiricamente $\approx n^{1{,}25}$) | $O(n^{1{,}5})$ | $O(1)$ | Não | Sim |
| **Shellsort (Ciura)** | $O(n \log n)$ | em aberto | desconhecido | $O(1)$ | Não | Sim |
| **Merge Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | Sim | Não |
| **Quicksort (último)** | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | Não | Sim |
| **Quicksort (aleatório)** | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | Não | Sim |

<a id="parteC"></a>
# Parte C — Algoritmos de tempo linear (não comparativos)

📘 **Por que é possível ordenar em tempo linear?** Todo algoritmo baseado **apenas em comparações** pode ser modelado por uma **árvore de decisão** com pelo menos $n!$ folhas, cuja altura é $\Omega(\log n!) = \Omega(n \log n)$. Portanto, nenhum algoritmo comparativo tem pior caso melhor que $\Omega(n \log n)$.

Os algoritmos desta parte **não comparam elementos entre si**: eles usam os **valores** como índices ou para calcular posições. Por isso escapam do limite inferior — mas pagam com **hipóteses sobre a entrada** (inteiros num intervalo limitado, dígitos, distribuição uniforme). Entender essas hipóteses é o ponto central desta parte.

## C.1 Counting Sort 📘

**Hipótese.** As chaves são inteiros em $[0, k)$.

**Ideia.**
1. Conta quantas vezes cada valor aparece (vetor `cont` de tamanho $k$).
2. Transforma `cont` em **somas prefixas**: `cont[x]` passa a ser o número de elementos $\le x$, ou seja, a posição *final* (+1) do último $x$.
3. Percorre a entrada **de trás para frente**, colocando cada elemento em `saida[cont[x]-1]` e decrementando `cont[x]`. Percorrer de trás para frente é o que torna o algoritmo **estável**.

**Pseudocódigo**
```
COUNTING-SORT(v, k)
  cont ← vetor de k zeros
  para cada x em v: cont[x] ← cont[x] + 1
  para x ← 1 até k-1: cont[x] ← cont[x] + cont[x-1]
  saida ← vetor de tamanho n
  para i ← n-1 até 0 (decrescente)
      x ← v[i]
      cont[x] ← cont[x] - 1
      saida[cont[x]] ← x                ▹ conta 1 movimentação
  retorne saida
```

**Implementação.** Calcule $k$ como `max(v) + 1` com um laço simples (não use `max()` pronto? — **pode usar**, `max` não ordena). Para permitir o uso no Radix Sort, implemente também `counting_sort_por_chave(v, k, chave, c)`, que ordena pelo valor `chave(x)` em vez de `x`.

**Complexidade:** $\Theta(n + k)$ de tempo e de espaço. Quando $k = O(n)$, é linear.

In [ ]:
# 🛠️ Counting Sort
def counting_sort_por_chave(v, k, chave, c):
    # ordenação estável por chave(x), com chave(x) em [0, k)
    cont = [0] * k
    for x in v:
        cont[chave(x)] += 1
    for i in range(1, k):
        cont[i] += cont[i - 1]

    saida = [None] * len(v)
    for i in range(len(v) - 1, -1, -1):   # de trás para frente: mantém estabilidade
        d = chave(v[i])
        cont[d] -= 1
        escrever(saida, cont[d], v[i], c)
    return saida

def counting_sort(v, c):
    if not v:
        return v
    k = max(v) + 1
    return counting_sort_por_chave(v, k, lambda x: x, c)

## C.2 Radix Sort (LSD) 📘

**Hipótese.** As chaves são inteiros não negativos com no máximo $d$ dígitos na base $b$.

**Ideia.** Ordena pelos dígitos, **do menos significativo para o mais significativo** (LSD — *Least Significant Digit*), usando a cada passada um algoritmo **estável** (o Counting Sort com $k = b$). A estabilidade é essencial: ela preserva a ordem obtida nas passadas anteriores.

**Pseudocódigo**
```
RADIX-SORT(v, b)
  M ← maior elemento de v
  exp ← 1
  enquanto M // exp > 0
      v ← COUNTING-SORT-POR-CHAVE(v, k=b, chave(x) = (x // exp) mod b)
      exp ← exp · b
  retorne v
```

**Complexidade:** $\Theta(d\,(n + b))$, com $d = \lfloor \log_b M \rfloor + 1$.

**Escolha da base.** Implemente com base padrão $b = 2^8 = 256$ e permita outras bases pelo parâmetro. Na discussão você analisará o efeito de $b$: base maior → menos passadas, mas vetor de contagem maior. (Resultado clássico: escolher $b \approx n$ dá $d = O(\log M / \log n)$ passadas.)

In [ ]:
# 🛠️ Radix Sort LSD
def radix_sort(v, c, base=256):
    if len(v) < 2:
        return v
    minimo = min(v)                 # desloca pelo mínimo para aceitar negativos
    amplitude = max(v) - minimo
    exp = 1
    while amplitude // exp > 0:
        v = counting_sort_por_chave(v, base, lambda x: ((x - minimo) // exp) % base, c)
        exp *= base
    return v

def radix_sort_base10(v, c): return radix_sort(v, c, base=10)

## C.3 Bucket Sort 📘

**Hipótese.** As chaves são reais em $[0, 1)$ **distribuídos uniformemente**.

**Ideia.**
1. Cria $n$ baldes (listas vazias).
2. Coloca cada elemento $x$ no balde $\lfloor n \cdot x \rfloor$.
3. Ordena cada balde com **Insertion Sort** (baldes costumam ser pequenos).
4. Concatena os baldes em ordem.

**Pseudocódigo**
```
BUCKET-SORT(v)
  n ← |v|
  B ← n listas vazias
  para cada x em v: insira x em B[⌊n·x⌋]
  para i ← 0 até n-1: INSERTION-SORT(B[i])
  retorne B[0] ++ B[1] ++ ... ++ B[n-1]
```

**Análise.** Seja $n_i$ o tamanho do balde $i$. O tempo é $\Theta(n) + \sum_i O(n_i^2)$. Sob distribuição uniforme, $n_i \sim \text{Binomial}(n, 1/n)$ e $E[n_i^2] = 2 - 1/n$, logo o **caso médio é $\Theta(n)$**. No **pior caso** (todos no mesmo balde) o custo é $\Theta(n^2)$.

**Instrumentação.** Aqui as comparações vêm do Insertion Sort nos baldes; as movimentações incluem inserções nos baldes e as escritas na saída.

In [ ]:
# 🛠️ Bucket Sort
def bucket_sort(v, c):
    n = len(v)
    if n < 2:
        return v
    baldes = [[] for _ in range(n)]
    for x in v:
        baldes[min(int(x * n), n - 1)].append(x)
        c.trocas += 1               # inserção no balde conta 1 movimentação

    k = 0
    for b in baldes:
        insertion_sort_intervalo(b, 0, len(b) - 1, c)
        for x in b:
            escrever(v, k, x, c)
            k += 1
    return v

## C.4 Análise teórica dos lineares ✍️

### 1. Demonstração do limite inferior $\Omega(n \log n)$ para ordenação por comparação

Qualquer algoritmo de ordenação baseado exclusivamente em comparações de elementos por pares pode ser representado por uma **árvore de decisão**, na qual cada nó interno representa uma comparação $x_i \le x_j$ e cada folha representa uma permutação final dos $n$ elementos da entrada.

- Um vetor de tamanho $n$ possui $n!$ permutações possíveis.
- Para que o algoritmo ordene corretamente qualquer entrada, a árvore de decisão precisa ter **pelo menos $n!$ folhas** (uma para cada ordem possível).
- Uma árvore binária de altura $h$ tem no máximo $2^h$ folhas. Logo:
  $$2^h \ge n! \implies h \ge \log_2 (n!)$$

Para estimar $\log_2 (n!)$, aplicamos uma limitação inferior simples considerando apenas os maiores $n/2$ termos do fatorial:
$$n! = 1 \cdot 2 \cdots n \ge \left(\frac{n}{2}\right)^{n/2}$$

Aplicando o logaritmo em ambos os lados:
$$\log_2 (n!) \ge \log_2 \left(\frac{n}{2}\right)^{n/2} = \frac{n}{2} \log_2 \left(\frac{n}{2}\right) = \frac{n}{2} (\log_2 n - 1) = \Omega(n \log n)$$

Portanto, a altura $h$ da árvore (que representa o número de comparações no pior caso) é limitada por $h = \Omega(n \log n)$. *(Referência: Cormen et al., Introduction to Algorithms, Capítulo 8)*.

---

### 2. Tabela comparativa e violações de hipótese

| Algoritmo | Melhor | Médio | Pior | Espaço extra | Estável? | Hipótese sobre a entrada |
|---|---|---|---|---|---|---|
| **Counting Sort** | $\Theta(n + k)$ | $\Theta(n + k)$ | $\Theta(n + k)$ | $\Theta(n + k)$ | Sim | Chaves são inteiros em uma faixa limitada $[0, k-1]$. |
| **Radix Sort (base $b$)** | $\Theta(d(n + b))$ | $\Theta(d(n + b))$ | $\Theta(d(n + b))$ | $\Theta(n + b)$ | Sim | Chaves compostas por $d$ dígitos em uma base finita $b$ (não negativos). |
| **Bucket Sort** | $\Theta(n + k)$ | $\Theta(n)$ | $\Theta(n^2)$ | $\Theta(n)$ | Sim | Chaves são reais uniformemente distribuídos no intervalo $[0, 1)$. |

#### O que acontece quando a hipótese é violada?

* **Counting Sort com $k = n^2$:** 
  O vetor de frequências $C$ exige alocação de memória de tamanho $O(n^2)$ e precisa ser inicializado com zeros, elevando o tempo de execução para $\Theta(n + n^2) = \Theta(n^2)$. O algoritmo perde totalmente sua vantagem linear e causa alto uso de memória/RAM.
* **Bucket Sort com dados concentrados:** 
  Se a distribuição não for uniforme e todos os $n$ elementos caírem no mesmo balde, a ordenação interna desse balde usando Insertion Sort exigirá tempo $\Theta(n^2)$ no pior caso.
* **Radix Sort com números negativos:** 
  O cálculo de índices por módulo/divisão de dígitos ou pelo Counting Sort subjacente falha com valores negativos (índices fora do intervalo). Para tratar negativos, é necessário fazer um deslocamento de offset prévio (shifting) ou ordenar positivos e negativos separadamente.

---

### 3. Impacto da instabilidade no Radix Sort (com contraexemplo)

O Radix Sort ordena os elementos dígito a dígito, do **dígito menos significativo (LSD)** para o mais significativo (MSD). O algoritmo depende criticamente da **estabilidade** do algoritmo interno para garantir que, ao ordenar por um dígito mais significativo, a ordem relativa conquistada pelos dígitos menos significativos seja estritamente mantida para dígitos iguais.

#### Contraexemplo (sub-rotina instável):
Considere ordenar as duas chaves de dois dígitos: **`[21, 20]`** usando base 10.

1. **Passo 1 (Dígito menos significativo - Unidades):**
   - Comparando as unidades (`1` de 21, `0` de 20): `20` vem antes de `21`.
   - Vetor resultante: **`[20, 21]`**.
2. **Passo 2 (Dígito mais significativo - Dezenas):**
   - Ambas as dezenas são iguais a `2`.
   - Suponha que seja usada uma sub-rotina **instável** (ex.: Quicksort instável). Ao processar o balde da dezena `2`, a sub-rotina pode inverter a ordem relativa existente entre `20` e `21`.
   - Vetor resultante após o Passo 2: **`[21, 20]`** $\implies$ **FALHA** (o vetor volta a ficar desordenado).

---

### 4. Dedução da expectativa $E[n_i^2] = 2 - 1/n$ no Bucket Sort

No Bucket Sort com $n$ elementos e $n$ baldes idênticos distribuídos uniformemente no intervalo $[0, 1)$, a variável aleatória $n_i$ representa o número de elementos que caem no balde $i$.

1. Como cada elemento tem probabilidade $p = 1/n$ de cair em um balde específico e cada inserção é independente, a variável $n_i$ segue uma **Distribuição Binomial**:
   $$n_i \sim \text{Binomial}(n, p), \quad \text{onde } p = \frac{1}{n}$$

2. Para uma distribuição binomial $X \sim \text{Binomial}(n, p)$, a média e a variação são dadas por:
   $$E[X] = n \cdot p$$
   $$\text{Var}[X] = n \cdot p \cdot (1 - p)$$

3. Substituindo $p = 1/n$:
   $$E[n_i] = n \cdot \left(\frac{1}{n}\right) = 1$$
   $$\text{Var}[n_i] = n \cdot \left(\frac{1}{n}\right) \cdot \left(1 - \frac{1}{n}\right) = 1 - \frac{1}{n}$$

4. Utilizando a identidade do momento de segundo grau $E[X^2] = \text{Var}[X] + (E[X])^2$:
   $$E[n_i^2] = \text{Var}[n_i] + (E[n_i])^2$$
   $$E[n_i^2] = \left(1 - \frac{1}{n}\right) + 1^2$$
   $$E[n_i^2] = 2 - \frac{1}{n}$$

Esta expectativa demonstra por que o tempo esperado do Bucket Sort para $n$ baldes é $O(n)$:
$$E[T(n)] = \Theta(n) + \sum_{i=0}^{n-1} O(E[n_i^2]) = \Theta(n) + n \cdot O\left(2 - \frac{1}{n}\right) = \Theta(n)$$

<a id="parteD"></a>
# Parte D — Verificação de corretude

A célula abaixo registra todos os algoritmos e roda uma bateria de testes com casos-limite (vetor vazio, um elemento, repetidos, negativos não entram nos lineares, etc.). **Todos os testes devem passar antes de você iniciar os experimentos.**

Em seguida, **acrescente pelo menos 3 testes próprios** na célula indicada — por exemplo, um teste de **estabilidade** usando pares `(chave, rótulo)`, um teste para `insertion_sort_intervalo` ordenando apenas um trecho, ou um teste que confira o número **exato** de comparações do Selection Sort ($n(n-1)/2$).

In [ ]:
# 🧰 Registro dos algoritmos (não altere os nomes)
# grupo: "quadratico", "nlogn" ou "linear"; dominio: "int" (inteiros >= 0) ou "float01"
ALGORITMOS = {
    "bubble":           dict(f=bubble_sort,         grupo="quadratico", dominio="int"),
    "insertion":        dict(f=insertion_sort,      grupo="quadratico", dominio="int"),
    "selection":        dict(f=selection_sort,      grupo="quadratico", dominio="int"),
    "shell_ciura":      dict(f=shellsort_ciura,     grupo="nlogn",      dominio="int"),
    "shell_knuth":      dict(f=shellsort_knuth,     grupo="nlogn",      dominio="int"),
    "shell_shell":      dict(f=shellsort_shell,     grupo="nlogn",      dominio="int"),
    "merge":            dict(f=merge_sort,          grupo="nlogn",      dominio="int"),
    "quick_aleatorio":  dict(f=quicksort_aleatorio, grupo="nlogn",      dominio="int"),
    "quick_mediana3":   dict(f=quicksort_mediana3,  grupo="nlogn",      dominio="int"),
    "quick_ultimo":     dict(f=quicksort_ultimo,    grupo="nlogn",      dominio="int"),
    "counting":         dict(f=counting_sort,       grupo="linear",     dominio="int"),
    "radix":            dict(f=radix_sort,          grupo="linear",     dominio="int"),
    "radix_b10":        dict(f=radix_sort_base10,   grupo="linear",     dominio="int"),
    "bucket":           dict(f=bucket_sort,         grupo="linear",     dominio="float01"),
}

In [ ]:
# 🧰 Bateria de testes
def casos_de_teste(dominio):
    if dominio == "float01":
        rng = random.Random(7)
        return [[], [0.5], [0.3, 0.1], [0.2, 0.2, 0.2],
                [0.0, 0.999, 0.5, 0.25, 0.75],
                [rng.random() for _ in range(200)],
                gerar_floats(300, seed=3, distribuicao="concentrada")]
    casos = [[], [5], [2, 1], [1, 2], [3, 3, 3], [0, 0, 1, 0],
             [5, 4, 3, 2, 1], [1, 2, 3, 4, 5], [10, 0, 256, 255, 257, 65536, 1]]
    for t in TIPOS:
        for n in (17, 100):
            casos.append(gerar_inteiros(t, n, seed=n))
    casos.append(gerar_inteiros("aleatoria", 300, seed=9, k=5))       # muitos repetidos
    casos.append(gerar_inteiros("aleatoria", 300, seed=9, k=10**6))   # valores grandes
    return casos


def testar(nome):
    info = ALGORITMOS[nome]
    try:
        for caso in casos_de_teste(info["dominio"]):
            entrada = list(caso)
            c = Contador()
            saida = info["f"](entrada, c)
            gabarito = sorted(caso)                      # sorted() permitido SÓ aqui
            if saida != gabarito:
                return f"FALHOU  em {caso[:10]}{'...' if len(caso) > 10 else ''} -> {saida[:10] if saida else saida}"
        return "ok"
    except NotImplementedError:
        return "não implementado"
    except Exception as e:
        return f"ERRO   {type(e).__name__}: {e}"


def testar_todos():
    for nome in ALGORITMOS:
        print(f"{nome:>16}: {testar(nome)}")

testar_todos()

In [ ]:
# 🛠️ Testes próprios
from itertools import permutations

ESTAVEIS = ["bubble", "insertion", "merge", "counting", "radix", "radix_b10", "bucket"]

class Item:
    def __init__(self, chave, rotulo): self.chave, self.rotulo = chave, rotulo
    def __lt__(self, o): return self.chave < o.chave
    def __le__(self, o): return self.chave <= o.chave
    def __gt__(self, o): return self.chave > o.chave
    def __repr__(self): return f"{self.chave}{self.rotulo}"

# Counting e Radix usam o valor como índice e o Bucket multiplica por n: precisam de int/float com rótulo
class IntRotulado(int):
    def __new__(cls, valor, rotulo):
        x = int.__new__(cls, valor); x.rotulo = rotulo; return x

class FloatRotulado(float):
    def __new__(cls, valor, rotulo):
        x = float.__new__(cls, valor); x.rotulo = rotulo; return x

def chave(x):
    return getattr(x, "chave", x)

def rotular(nome, chaves):
    info = ALGORITMOS[nome]
    if info["dominio"] == "float01":
        return [FloatRotulado(k, i) for i, k in enumerate(chaves)]
    if info["grupo"] == "linear":
        return [IntRotulado(k, i) for i, k in enumerate(chaves)]
    return [Item(k, i) for i, k in enumerate(chaves)]


def teste_estabilidade():
    # chaves com muitos repetidos; o rótulo é a posição original
    rng = random.Random(123)
    for nome in ESTAVEIS:
        if ALGORITMOS[nome]["dominio"] == "float01":
            chaves = [rng.choice([0.1, 0.15, 0.5, 0.9]) for _ in range(300)]
        else:
            chaves = [rng.randrange(10) for _ in range(300)]
        saida = ALGORITMOS[nome]["f"](rotular(nome, chaves), Contador())
        for a, b in zip(saida, saida[1:]):
            assert chave(a) <= chave(b), f"{nome}: saída fora de ordem"
            assert chave(a) < chave(b) or a.rotulo < b.rotulo, \
                f"{nome} não é estável: na chave {chave(a)}, o rótulo {a.rotulo} ficou antes do {b.rotulo}"
    print("estabilidade: ok", ESTAVEIS)


def teste_vetor_ordenado():
    n = 200
    for nome, info in ALGORITMOS.items():
        if info["dominio"] == "float01":
            entradas = [[i / n for i in range(n)], [(i // 10) / n for i in range(n)]]
        else:
            entradas = [gerar_inteiros("ordenada", n, seed=1), gerar_inteiros("ordenada", n, seed=1, k=20)]
        for v in entradas:
            saida = info["f"](list(v), Contador())
            assert saida == v, f"{nome} alterou um vetor já ordenado"

    # contagens exatas em vetor ordenado sem repetidos
    v = gerar_inteiros("ordenada", n, seed=1)
    def conta(nome):
        c = Contador(); ALGORITMOS[nome]["f"](list(v), c); return c.comp, c.trocas
    assert conta("bubble") == (n - 1, 0), f"Bubble: esperado {(n - 1, 0)}, obtido {conta('bubble')}"
    assert conta("insertion") == (n - 1, 0), f"Insertion: esperado {(n - 1, 0)}, obtido {conta('insertion')}"
    assert conta("selection") == (n * (n - 1) // 2, 0), f"Selection: esperado {(n * (n - 1) // 2, 0)}, obtido {conta('selection')}"
    for nome in ("shell_shell", "shell_knuth", "shell_ciura"):
        assert conta(nome)[1] == 0, f"{nome}: não deveria mover elementos em vetor ordenado"
    assert conta("quick_ultimo")[0] == n * (n - 1) // 2, \
        f"Quick-último: esperado n(n-1)/2 = {n * (n - 1) // 2} comparações (pior caso), obtido {conta('quick_ultimo')[0]}"
    print("vetor ordenado: ok")


def teste_casos_de_borda():
    pequenos = [[], [0], [1, 0], [0, 1], [1, 1], [2, 2, 2], [0, 0, 1, 0], [1000, 0, 999]]
    pequenos += [list(p) for p in permutations([0, 1, 2])]
    pequenos += [list(p) for p in sorted(set(permutations([0, 1, 1, 2])))]
    for nome, info in ALGORITMOS.items():
        for caso in pequenos:
            entrada = [x / 1001 for x in caso] if info["dominio"] == "float01" else list(caso)
            saida = info["f"](list(entrada), Contador())
            assert saida == sorted(entrada), f"{nome} falhou em {entrada}: devolveu {saida}"
    print(f"casos de borda: ok ({len(pequenos)} vetores pequenos por algoritmo)")


teste_estabilidade()
teste_vetor_ordenado()
teste_casos_de_borda()

<a id="parteE"></a>
# Parte E — Análise teórica consolidada ✍️

| Algoritmo | Melhor | Médio | Pior | Espaço extra | Estável | In-place | Recorrência / observação |
|---|---|---|---|---|---|---|---|
| **Bubble Sort** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | Sim | Sim | $T(n) = T(n-1) + O(n)$ |
| **Insertion Sort** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | Sim | Sim | $T(n) = T(n-1) + O(n)$ |
| **Selection Sort** | $O(n^2)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | Não | Sim | $T(n) = T(n-1) + O(n)$ |
| **Shellsort (Ciura)** | $O(n \log n)$ | em aberto | desconhecido | $O(1)$ | Não | Sim | Sequência estática: $[1, 4, 10, 23, 57, 132, 301, 701, \dots]$ |
| **Merge Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | Sim | Não | $T(n) = 2T(n/2) + \Theta(n)$ |
| **Quicksort (aleatório)** | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | Não | Sim | $T(n) = 2T(n/2) + \Theta(n)$ (caso esperado) |
| **Counting Sort** | $O(n + k)$ | $O(n + k)$ | $O(n + k)$ | $O(n + k)$ | Sim | Não | Não comparativo; $k = \max(A) + 1$ (chaves em $[0, k)$) |
| **Bucket Sort** | $O(n)$ | $O(n)$ | $O(n^2)$ | $O(n)$ | Sim | Não | Requer distribuição uniforme no caso médio |
| **Radix Sort** | $O(d(n + b))$ | $O(d(n + b))$ | $O(d(n + b))$ | $O(n + b)$ | Sim | Não | $d = \lfloor \log_b (M - m) \rfloor + 1$ passagens do Counting Sort em base $b$ ($M$ e $m$: maior e menor valor) |

---

### Previsões Teóricas e Expoentes Esperados ($\alpha$)

#### 1. Desempenho Esperado por Tipo de Entrada
* **Entradas Aleatórias:**
  * **Mais rápidos:** Algoritmos não comparativos (Counting Sort ou Radix Sort) para intervalos $k$ contidos em $O(n)$, seguidos de perto pelo **Quicksort (aleatório)** e **Merge Sort**.
  * **Mais lentos:** **Bubble Sort** e **Selection Sort**, devido ao custo estritamente quadrático $O(n^2)$.

* **Entradas Ordenadas / Quase Ordenadas:**
  * **Mais rápidos:** **Insertion Sort** e **Bubble Sort (com flag de parada)**, atingindo complexidade estritamente linear $O(n)$ com constantes extremamente baixas, superando até mesmo os algoritmos $O(n \log n)$.
  * **Mais lentos:** **Selection Sort**, pois realiza obrigatoriamente todas as $\frac{n(n-1)}{2}$ comparações, independente da ordenação prévia do vetor.

* **Entradas Inversamente Ordenadas:**
  * **Mais rápidos:** **Merge Sort** e **Quicksort (aleatório)**, que mantêm a complexidade $O(n \log n)$ devido à divisão balanceada ou ao pivô sorteado.
  * **Mais lentos:** **Bubble Sort** e **Insertion Sort**, que realizam o número máximo de trocas e comparações possíveis ($O(n^2)$).

---

#### 2. Expoente Assintótico Esperado ($\alpha$)
Considerando a relação de ajuste empírico $T(n) \approx c \cdot n^\alpha$, o comportamento do expoente $\alpha = \frac{\log T(n)}{\log n}$ é previsto da seguinte forma para grandes valores de $n$:

* **Algoritmos Quadráticos ($\alpha \approx 2.0$):**
  * **Selection Sort:** $\alpha \approx 2.0$ em **todas** as entradas.
  * **Bubble Sort e Insertion Sort:** $\alpha \approx 2.0$ no caso médio e pior caso (entradas aleatórias ou inversas); $\alpha \approx 1.0$ no melhor caso (entradas ordenadas).

* **Algoritmos subquadráticos ($\alpha \approx 1.20 \text{ a } 1.33$):**
  * **Shellsort (Ciura):** $\alpha \approx 1.20 \text{ a } 1.33$ (estimativa empírica; não há limite demonstrado para essa sequência).

* **Algoritmos Quase Lineares ($\alpha \approx 1.0 - 1.15$):**
  * **Merge Sort e Quicksort (aleatório):** Como a complexidade teórica é $O(n \log n)$, a inclinação log-log ajustada em intervalos limitados de $n$ gera um $\alpha$ ligeiramente superior a $1.0$, situando-se tipicamente entre **$1.05$ e $1.15$**.

* **Algoritmos Lineares ($\alpha \approx 1.0$):**
  * **Counting Sort, Radix Sort e Bucket Sort (uniforme):** $\alpha \approx 1.0$, desde que a variação do parâmetro de faixa/número de dígitos ($k$ ou $d$) permaneça constante ou proporcional a $O(n)$.

<a id="parteF"></a>
# Parte F — Avaliação experimental

## F.1 Desenho dos experimentos 📘

Como Python é interpretado (cada operação elementar custa da ordem de dezenas de nanossegundos, e a instrumentação dobra esse custo), as faixas de $n$ são **calibradas pela complexidade** de cada grupo. Assim, cada experimento termina em tempo razoável e ainda cobre pelo menos **uma a duas ordens de grandeza** de $n$ — o necessário para enxergar o crescimento em escala log-log.

| Experimento | Algoritmos | Tipos de entrada | Valores de $n$ | Seeds |
|---|---|---|---|---|
| **E1 — Faixa comum** | todos os 9 (versões principais) | aleatória, ordenada, inversa, quase ordenada (Bucket: uniforme) | 100, 200, 500, 1 000, 2 000, 5 000 | 5 |
| **E2 — Quadráticos** | bubble, insertion, selection | os 4 tipos | 500, 1 000, 2 000, 5 000, 10 000 | 5 (3 para $n = 10^4$ se necessário) |
| **E3 — $O(n \log n)$ e Shellsort** | shell (3 gaps), merge, quick (3 pivôs) | os 4 tipos | $10^3$, $3\cdot10^3$, $10^4$, $3\cdot10^4$, $10^5$ (opcional: $3\cdot10^5$, $10^6$) | 5 |
| **E4 — Lineares** | counting, radix, bucket | aleatória com $k \in \{100,\ n,\ n^{1{,}5}\}$ (Bucket: uniforme e concentrada) | $10^3$, $10^4$, $10^5$, $3\cdot10^5$, $10^6$ | 5 |
| **E5 — Pivô do Quicksort** | quick_ultimo, quick_aleatorio, quick_mediana3 | ordenada, inversa, aleatória com $k = 10$ | 250, 500, 1 000, 2 000, 4 000 | 5 |
| **E6 — Radix: efeito da base** | radix (base 10, 256, 65 536) | aleatória com $k = 10^9$ | $10^4$, $10^5$, $10^6$ | 5 |

> ⚠️ **Mecanismo de proteção:** a função `rodar_experimento` recebe `tempo_limite` (segundos). Se uma execução (ou a média das seeds) passar desse limite, ou se a previsão para o próximo tamanho — supondo crescimento quadrático a partir do tamanho anterior — ultrapassar 4× o limite, os tamanhos seguintes são **pulados** para esse algoritmo/tipo (e isso aparece no log). Nesses casos, o último tamanho medido pode ter menos seeds que o previsto. Isso é esperado para, por exemplo, `quick_ultimo` em entrada ordenada. **Documente** no relatório quais combinações foram puladas e por quê.

> ⏱️ **Estimativa de tempo:** o modo completo pode levar de 30 a 90 minutos, dependendo do computador. Use `MODO = "rapido"` enquanto desenvolve.

In [ ]:
# 🧰 Configuração do modo de execução
MODO = "completo"        # troque para "completo" para os resultados finais
SEEDS = [11, 22, 33, 44, 55] if MODO == "completo" else [11, 22]

def faixa(completo, rapido):
    return completo if MODO == "completo" else rapido

print("Modo:", MODO, "| seeds:", SEEDS)

## F.2 Executor de experimentos 🧰

`rodar_experimento` gera **um vetor por (tipo, n, seed)**, entrega **cópias** a cada algoritmo, mede tempo com `perf_counter` e o GC desligado, confere se a saída está ordenada e devolve um `DataFrame` com uma linha por execução. `resumir` calcula média e desvio padrão.

In [ ]:
# 🧰 Executor
def _gerar(dominio, tipo, n, seed, k):
    if dominio == "float01":
        dist = tipo if tipo in ("uniforme", "concentrada") else "uniforme"
        return gerar_floats(n, seed, dist)
    return gerar_inteiros(tipo, n, seed, k)


def rodar_experimento(nome_exp, algoritmos, tipos, ns, seeds=None, k_fn=None,
                      tempo_limite=None, verificar=True, funcoes=None, verbose=True):
    # algoritmos : lista de nomes em ALGORITMOS (ou em 'funcoes', se fornecido)
    # tipos      : lista de tipos de entrada
    # ns         : lista crescente de tamanhos
    # k_fn       : função n -> k (intervalo de valores); None usa k = n
    # funcoes    : dict opcional nome -> (f, dominio), para variantes fora do registro
    seeds = seeds or SEEDS
    if tempo_limite is None:
        tempo_limite = faixa(20.0, 3.0)
    registros = []
    for alg in algoritmos:
        if funcoes and alg in funcoes:
            f, dominio = funcoes[alg]
        else:
            f, dominio = ALGORITMOS[alg]["f"], ALGORITMOS[alg]["dominio"]
        nao_implementado = False
        for tipo in tipos:
            if nao_implementado:
                break
            n_ant, media_ant = None, None
            for n in ns:
                k = k_fn(n) if k_fn else None
                # Previsão conservadora (supõe crescimento quadrático): se o tamanho atual
                # deve estourar o limite, nem começa.
                if media_ant is not None and media_ant * (n / n_ant) ** 2 > 4 * tempo_limite:
                    if verbose: print(f"  [{alg}|{tipo}] previsão para n={n} excede o limite — pulando tamanhos maiores")
                    break
                tempos = []
                estourou = False
                try:
                    for seed in seeds:
                        base = _gerar(dominio, tipo, n, seed, k)
                        entrada = list(base)             # cópia: todos recebem o mesmo vetor
                        c = Contador()
                        saida = None                     # libera a saída anterior fora da medição
                        gc.collect(); gc.disable()
                        t0 = time.perf_counter()
                        saida = f(entrada, c)
                        t1 = time.perf_counter()
                        gc.enable()
                        if verificar and not esta_ordenado(saida):
                            raise AssertionError(f"{alg}: saída não ordenada (tipo={tipo}, n={n})")
                        tempos.append(t1 - t0)
                        registros.append(dict(experimento=nome_exp, algoritmo=alg, tipo=tipo,
                                              n=n, k=k, seed=seed, tempo=t1 - t0,
                                              comp=c.comp, trocas=c.trocas))
                        if t1 - t0 > tempo_limite:           # uma única execução já estourou
                            estourou = True
                            break
                except NotImplementedError:
                    gc.enable()
                    if verbose: print(f"  [{alg}] não implementado — pulando")
                    nao_implementado = True
                    break
                except RecursionError:
                    gc.enable()
                    if verbose: print(f"  [{alg}|{tipo}] RecursionError em n={n} — pulando tamanhos maiores")
                    break
                media = statistics.mean(tempos)
                if verbose:
                    print(f"  {alg:>15} | {tipo:>14} | n={n:>8} | {media*1000:10.2f} ms")
                n_ant, media_ant = n, media
                if estourou or media > tempo_limite:
                    if verbose: print(f"  [{alg}|{tipo}] média {media:.1f}s > limite — pulando tamanhos maiores")
                    break
    return pd.DataFrame(registros)


def resumir(df):
    # Média e desvio padrão por (experimento, algoritmo, tipo, n, k).
    if df.empty:
        return df
    chaves = ["experimento", "algoritmo", "tipo", "n", "k"]
    g = df.fillna({"k": -1}).groupby(chaves)[["tempo", "comp", "trocas"]]
    res = g.agg(["mean", "std"])
    res.columns = [f"{a}_{b}" for a, b in res.columns]
    return res.reset_index()

## F.3 Funções de gráfico e ajuste 🧰

- `plot_metrica` desenha média ± desvio padrão (barras de erro) para vários algoritmos, com escala linear ou log-log, e sobrepõe **curvas teóricas** escaladas para coincidir com o último ponto medido de um algoritmo de referência.
- `estimar_expoente` ajusta uma reta a $\log T$ versus $\log n$. A inclinação $\alpha$ estima o expoente em $T(n) \approx c\,n^\alpha$. Lembre-se: para $n \log n$, espera-se $\alpha$ ligeiramente acima de 1.
- `razao_teorica` calcula $T(n)/g(n)$. Se $T(n) = \Theta(g(n))$, a razão deve **se estabilizar** à medida que $n$ cresce — este é um teste empírico muito mais convincente do que "as curvas parecem parecidas".

In [ ]:
# 🧰 Gráficos e ajustes
TEORICAS = {
    "n":        lambda n: n,
    "n log n":  lambda n: n * np.log2(n),
    "n^1.5":    lambda n: n ** 1.5,
    "n^2":      lambda n: n ** 2,
}

def plot_metrica(res, metrica="tempo", algoritmos=None, tipo=None, experimento=None,
                 loglog=True, teoricas=("n log n", "n^2"), ref=None, titulo=None, ax=None):
    if res is None or res.empty:
        print("Sem resultados para plotar (os algoritmos já foram implementados?).")
        return
    d = res.copy()
    if experimento: d = d[d.experimento == experimento]
    if tipo:        d = d[d.tipo == tipo]
    if algoritmos:  d = d[d.algoritmo.isin(algoritmos)]
    if d.empty:
        print("Nada para plotar (verifique filtros ou se os algoritmos foram implementados).")
        return
    ax = ax or plt.subplots()[1]
    for alg, g in d.groupby("algoritmo"):
        g = g.sort_values("n")
        ax.errorbar(g.n, g[f"{metrica}_mean"], yerr=g[f"{metrica}_std"].fillna(0),
                    marker="o", capsize=3, label=alg)
    if teoricas:
        ref = ref or d.algoritmo.iloc[0]
        gref = d[d.algoritmo == ref].sort_values("n")
        n_ult, y_ult = gref.n.iloc[-1], gref[f"{metrica}_mean"].iloc[-1]
        ns = np.linspace(d.n.min(), d.n.max(), 200)
        for nome in teoricas:
            fn = TEORICAS[nome]
            ax.plot(ns, fn(ns) * (y_ult / fn(n_ult)), "--", alpha=0.6, label=f"∝ {nome} (ajust. a {ref})")
    if loglog:
        ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("n")
    ax.set_ylabel({"tempo": "tempo (s)", "comp": "comparações", "trocas": "movimentações"}[metrica])
    ax.set_title(titulo or f"{metrica} — {tipo or ''} {'(log-log)' if loglog else ''}")
    ax.legend(fontsize=8)
    return ax


def estimar_expoente(res, metrica="tempo", experimento=None, n_min=0):
    # Tabela com alpha estimado por (algoritmo, tipo), usando pontos com n >= n_min.
    if res is None or res.empty:
        return pd.DataFrame()
    d = res.copy()
    if experimento: d = d[d.experimento == experimento]
    d = d[(d.n >= n_min) & (d[f"{metrica}_mean"] > 0)]
    linhas = []
    for (alg, tipo, k), g in d.groupby(["algoritmo", "tipo", "k"]):
        if g.n.nunique() >= 3:
            alpha, _ = np.polyfit(np.log(g.n), np.log(g[f"{metrica}_mean"]), 1)
            linhas.append(dict(algoritmo=alg, tipo=tipo, k=k, alpha=round(alpha, 3), pontos=g.n.nunique()))
    return pd.DataFrame(linhas)


def razao_teorica(res, algoritmo, tipo, g="n log n", metrica="tempo", experimento=None):
    if res is None or res.empty:
        return pd.DataFrame()
    d = res[(res.algoritmo == algoritmo) & (res.tipo == tipo)]
    if experimento: d = d[d.experimento == experimento]
    d = d.sort_values("n").copy()
    d[f"{metrica}/({g})"] = d[f"{metrica}_mean"] / TEORICAS[g](d.n.astype(float))
    return d[["n", f"{metrica}_mean", f"{metrica}/({g})"]]

## F.4 Execução dos experimentos

Execute uma célula por experimento. Cada uma salva um CSV em disco — se o kernel reiniciar, você pode recarregar com `pd.read_csv`.

In [ ]:
# E1 — Faixa comum a todos os algoritmos
PRINCIPAIS = ["bubble", "insertion", "selection", "shell_ciura", "merge",
              "quick_aleatorio", "counting", "radix", "bucket"]
ns_E1 = faixa([100, 200, 500, 1_000, 2_000, 5_000], [100, 200, 500, 1_000])

df_E1 = rodar_experimento("E1", PRINCIPAIS, TIPOS, ns_E1)
df_E1.to_csv("resultados_E1.csv", index=False)
res_E1 = resumir(df_E1)
res_E1.head(10)

In [ ]:
# E2 — Quadráticos
ns_E2 = faixa([500, 1_000, 2_000, 5_000, 10_000], [250, 500, 1_000])
df_E2 = rodar_experimento("E2", ["bubble", "insertion", "selection"], TIPOS, ns_E2, tempo_limite=faixa(60, 5))
df_E2.to_csv("resultados_E2.csv", index=False)
res_E2 = resumir(df_E2)

In [ ]:
# E3 — O(n log n) e Shellsort
ns_E3 = faixa([1_000, 3_000, 10_000, 30_000, 100_000], [1_000, 3_000, 10_000])
# Opcional (computador rápido): acrescente 300_000 e 1_000_000 à lista acima.
ALG_E3 = ["shell_shell", "shell_knuth", "shell_ciura", "merge",
          "quick_aleatorio", "quick_mediana3", "quick_ultimo"]
df_E3 = rodar_experimento("E3", ALG_E3, TIPOS, ns_E3, tempo_limite=faixa(20, 2))
df_E3.to_csv("resultados_E3.csv", index=False)
res_E3 = resumir(df_E3)

In [ ]:
# E4 — Lineares, variando k
ns_E4 = faixa([1_000, 10_000, 100_000, 300_000, 1_000_000], [1_000, 10_000, 50_000])
frames = []
for rotulo, k_fn in [("k=100", lambda n: 100),
                     ("k=n",   lambda n: n),
                     ("k=n^1.5", lambda n: int(n ** 1.5))]:
    print(">>>", rotulo)
    d = rodar_experimento(f"E4 {rotulo}", ["counting", "radix"], ["aleatoria"], ns_E4, k_fn=k_fn)
    frames.append(d)
# Bucket Sort: distribuição uniforme x concentrada
frames.append(rodar_experimento("E4 bucket", ["bucket"], ["uniforme", "concentrada"], ns_E4))
# Referência comparativa na mesma faixa (limitada para não demorar)
frames.append(rodar_experimento("E4 k=n", ["merge", "quick_aleatorio"], ["aleatoria"], ns_E4, tempo_limite=faixa(15, 2)))
df_E4 = pd.concat(frames, ignore_index=True)
df_E4.to_csv("resultados_E4.csv", index=False)
res_E4 = resumir(df_E4)

In [ ]:
# E5 — Impacto da escolha do pivô no Quicksort
ns_E5 = faixa([250, 500, 1_000, 2_000, 4_000], [250, 500, 1_000])
df_E5 = pd.concat([
    rodar_experimento("E5", ["quick_ultimo", "quick_aleatorio", "quick_mediana3"],
                      ["ordenada", "inversa"], ns_E5, tempo_limite=faixa(20, 2)),
    rodar_experimento("E5 k=10", ["quick_ultimo", "quick_aleatorio", "quick_mediana3"],
                      ["aleatoria"], ns_E5, k_fn=lambda n: 10, tempo_limite=faixa(20, 2)),
], ignore_index=True)
df_E5.to_csv("resultados_E5.csv", index=False)
res_E5 = resumir(df_E5)

In [ ]:
# E6 — Radix Sort: efeito da base
ns_E6 = faixa([10_000, 100_000, 1_000_000], [1_000, 10_000, 50_000])
variantes = {
    "radix_b10":    (lambda v, c: radix_sort(v, c, base=10),     "int"),
    "radix_b256":   (lambda v, c: radix_sort(v, c, base=256),    "int"),
    "radix_b65536": (lambda v, c: radix_sort(v, c, base=65_536), "int"),
}
df_E6 = rodar_experimento("E6", list(variantes), ["aleatoria"], ns_E6,
                          k_fn=lambda n: 10**9, funcoes=variantes)
df_E6.to_csv("resultados_E6.csv", index=False)
res_E6 = resumir(df_E6)

## F.5 Resultados: tabelas e gráficos

Abaixo há **exemplos** de como gerar os gráficos. Os gráficos **mínimos obrigatórios** estão listados; você pode (e deve) criar outros que ajudem sua análise. **Todo gráfico precisa de título, rótulos nos eixos com unidade e legenda**, e deve ser **comentado** em texto logo abaixo.

**Gráficos obrigatórios:**
1. **G1** — E1: tempo × $n$ dos 9 algoritmos em entrada aleatória, escala linear **e** log-log.
2. **G2** — E2: tempo × $n$ (log-log) dos quadráticos para cada tipo de entrada (4 subgráficos), com a curva $n^2$.
3. **G3** — E2: comparações e trocas × $n$ dos quadráticos em entrada aleatória, sobrepondo $n^2/2$ e $n^2/4$.
4. **G4** — E3: tempo × $n$ (log-log) de merge, quick_aleatorio e shell_ciura com as curvas $n \log n$ e $n^{1,5}$.
5. **G5** — E3: comparação das três sequências de gaps do Shellsort (tempo e comparações).
6. **G6** — E4: lineares com os três valores de $k$, com merge/quick como referência.
7. **G7** — E5: pivôs do Quicksort em entrada ordenada e com $k = 10$.
8. **G8** — Zoom em $n$ pequeno (E1, $n \le 1\,000$): insertion × merge × quick, mostrando onde as curvas se cruzam.
9. **Tabela T1** — expoentes $\alpha$ estimados (`estimar_expoente`) para todos os experimentos, comparados com o esperado.
10. **Tabela T2** — razões $T(n)/g(n)$ (`razao_teorica`) para pelo menos 4 algoritmos.

In [ ]:
# Exemplo — G1: 9 algoritmos, entrada aleatória (linear e log-log)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_metrica(res_E1, "tempo", tipo="aleatoria", loglog=False, teoricas=None,
             titulo="E1 — tempo (escala linear), entrada aleatória", ax=axes[0])
plot_metrica(res_E1, "tempo", tipo="aleatoria", loglog=True, teoricas=("n", "n log n", "n^2"),
             ref="merge", titulo="E1 — tempo (log-log), entrada aleatória", ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# Exemplo — G2: quadráticos em cada tipo de entrada
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, tipo in zip(axes.flat, TIPOS):
    plot_metrica(res_E2, "tempo", tipo=tipo, teoricas=("n", "n^2"), ref="selection",
                 titulo=f"E2 — quadráticos, {tipo}", ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Exemplo — Tabela T1: expoentes estimados
_nao_vazios = [r for r in [res_E1, res_E2, res_E3, res_E4, res_E5, res_E6] if not r.empty]
todos = pd.concat(_nao_vazios, ignore_index=True) if _nao_vazios else pd.DataFrame()
estimar_expoente(todos, "tempo", n_min=1_000)

In [ ]:
# Exemplo — Tabela T2: T(n)/(n log n) para o Merge Sort em entrada aleatória
razao_teorica(res_E3, "merge", "aleatoria", g="n log n", metrica="comp")

In [ ]:
# G3 — E2: comparações e movimentações dos quadráticos (aleatória), com n²/2 e n²/4
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metrica, nome in [(axes[0], "comp", "comparações"), (axes[1], "trocas", "movimentações")]:
    plot_metrica(res_E2, metrica, tipo="aleatoria", teoricas=None, ax=ax,
                 titulo=f"E2 — {nome} × n, entrada aleatória (log-log)")
    ns = np.array(sorted(res_E2.n.unique()), dtype=float)
    ax.plot(ns, ns ** 2 / 2, "--", alpha=0.6, label="n²/2")
    ax.plot(ns, ns ** 2 / 4, "--", alpha=0.6, label="n²/4")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# G4 — E3: merge, quick_aleatorio e shell_ciura com as curvas n log n e n^1,5
plot_metrica(res_E3, "tempo", algoritmos=["merge", "quick_aleatorio", "shell_ciura"], tipo="aleatoria",
             teoricas=("n log n", "n^1.5"), ref="merge", titulo="E3 — tempo × n, entrada aleatória (log-log)")
plt.show()

In [ ]:
# G5 — E3: as três sequências de gaps do Shellsort (tempo e comparações)
SHELLS = ["shell_shell", "shell_knuth", "shell_ciura"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_metrica(res_E3, "tempo", algoritmos=SHELLS, tipo="aleatoria", teoricas=None,
             titulo="E3 — Shellsort: tempo, entrada aleatória (log-log)", ax=axes[0])
plot_metrica(res_E3, "comp", algoritmos=SHELLS, tipo="aleatoria", teoricas=None,
             titulo="E3 — Shellsort: comparações, entrada aleatória (log-log)", ax=axes[1])
plot_metrica(res_E3, "tempo", algoritmos=SHELLS, tipo="ordenada", teoricas=None,
             titulo="E3 — Shellsort: tempo, entrada ordenada (log-log)", ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
# G6 — E4: lineares com os três valores de k, com merge e quick como referência
# rótulo por configuração: o mesmo algoritmo aparece em várias chamadas do E4
# o prefixo "ref." põe as referências por último na legenda e mantém suas cores entre os painéis
g6 = res_E4.copy()
config = g6.experimento.str.replace("E4 ", "", regex=False)
g6["algoritmo"] = np.where(g6.algoritmo == "bucket", "bucket " + g6.tipo,
                  np.where(g6.algoritmo.isin(["merge", "quick_aleatorio"]), "ref. " + g6.algoritmo + " (" + config + ")",
                           g6.algoritmo + " (" + config + ")"))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_metrica(g6, "tempo", algoritmos=["counting (k=100)", "counting (k=n)", "counting (k=n^1.5)", "ref. merge (k=n)"],
             teoricas=None, titulo="E4 — Counting Sort por k (log-log)", ax=axes[0])
plot_metrica(g6, "tempo", algoritmos=["radix (k=100)", "radix (k=n)", "radix (k=n^1.5)", "ref. merge (k=n)"],
             teoricas=None, titulo="E4 — Radix Sort (base 256) por k (log-log)", ax=axes[1])
plot_metrica(g6, "tempo", algoritmos=["bucket uniforme", "bucket concentrada", "ref. merge (k=n)", "ref. quick_aleatorio (k=n)"],
             teoricas=None, titulo="E4 — Bucket Sort e referências (log-log)", ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
# G7 — E5: pivôs do Quicksort em entrada ordenada, inversa e com k = 10
PIVOS = ["quick_ultimo", "quick_aleatorio", "quick_mediana3"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, exp, tipo, nome in [(axes[0], "E5", "ordenada", "entrada ordenada"),
                            (axes[1], "E5", "inversa", "entrada inversa"),
                            (axes[2], "E5 k=10", "aleatoria", "aleatória com k = 10")]:
    plot_metrica(res_E5, "tempo", algoritmos=PIVOS, tipo=tipo, experimento=exp, teoricas=("n log n", "n^2"),
                 ref="quick_ultimo", titulo=f"E5 — pivôs do Quicksort, {nome} (log-log)", ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# G8 — E1: insertion × merge × quick em n pequeno
ALG_G8 = ["insertion", "merge", "quick_aleatorio"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_metrica(res_E1[res_E1.n <= 1_000], "tempo", algoritmos=ALG_G8, tipo="aleatoria", loglog=False, teoricas=None,
             titulo="E1 — n ≤ 1 000, entrada aleatória", ax=axes[0])
plot_metrica(res_E1[res_E1.n <= 2_000], "tempo", algoritmos=ALG_G8, tipo="quase_ordenada", loglog=False, teoricas=None,
             titulo="E1 — n ≤ 2 000, entrada quase ordenada", ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# Gráfico adicional — E6: efeito da base no Radix Sort
plot_metrica(res_E6, "tempo", teoricas=("n",), ref="radix_b256", titulo="E6 — Radix Sort por base, k = 10⁹ (log-log)")
plt.show()

In [ ]:
# T1 — expoentes α do tempo por experimento, comparados com o esperado
# k recebe o nome do experimento: no E4, k varia com n e estimar_expoente separaria cada ponto
def alpha_esperado(linha):
    a, t, e = linha.algoritmo, linha.tipo, linha.experimento
    if a in ("bubble", "insertion"):
        return "1" if t == "ordenada" else "2"
    if a == "selection":
        return "2"
    if a.startswith("quick") and (e == "E5 k=10" or (a == "quick_ultimo" and t in ("ordenada", "inversa"))):
        return "2"
    if a == "merge" or a.startswith("quick"):
        return "≈ 1,1 (n log n)"
    if a.startswith("shell"):
        return "entre 1 e 1,5"
    if a == "counting" and e == "E4 k=n^1.5":
        return "1,5"
    return "1"

_nao_vazios = [r for r in [res_E1, res_E2, res_E3, res_E4, res_E5, res_E6] if not r.empty]
todos_exp = pd.concat(_nao_vazios, ignore_index=True)
T1 = estimar_expoente(todos_exp.assign(k=todos_exp.experimento), "tempo", n_min=1_000).rename(columns={"k": "experimento"})
if not T1.empty:
    T1["esperado"] = T1.apply(alpha_esperado, axis=1)
    T1 = T1[["experimento", "algoritmo", "tipo", "alpha", "esperado", "pontos"]]
T1

In [ ]:
# T2 — razões T(n)/g(n) para vários algoritmos
CASOS_T2 = [
    (res_E3, "merge",           "aleatoria", "n log n", "tempo", None),
    (res_E3, "merge",           "aleatoria", "n log n", "comp",  None),
    (res_E3, "quick_aleatorio", "aleatoria", "n log n", "comp",  None),
    (res_E3, "shell_ciura",     "aleatoria", "n^1.5",   "tempo", None),
    (res_E2, "insertion",       "aleatoria", "n^2",     "tempo", None),
    (res_E2, "selection",       "aleatoria", "n^2",     "comp",  None),
    (res_E4, "counting",        "aleatoria", "n",       "tempo", "E4 k=n"),
    (res_E4, "bucket",          "uniforme",  "n",       "tempo", "E4 bucket"),
]
for res, alg, tipo, g, metrica, exp in CASOS_T2:
    print(f"\n{alg} — {metrica}/({g}), entrada {tipo}" + (f", {exp}" if exp else ""))
    print(razao_teorica(res, alg, tipo, g=g, metrica=metrica, experimento=exp).to_string(index=False))

✍️ **Comentários sobre os resultados** (um parágrafo por gráfico/tabela, descrevendo o que se observa — sem ainda explicar as causas, o que fica para a Parte G):

G1. Na entrada aleatória, os algoritmos formam três faixas no gráfico log-log: os quadráticos, com inclinação próxima à de $n^2$; Shellsort, Merge e Quick, próximos de $n\log n$; e os lineares, os mais rápidos em todos os tamanhos. Na escala linear, os quadráticos se separam dos demais a partir de $n \approx 1\,000$. Em $n = 5\,000$, o Counting Sort levou 2,8 ms e o Bubble Sort 3,43 s.

G2. O Selection Sort segue $n^2$ nos quatro tipos de entrada, com tempos entre 7,59 s e 7,75 s em $n = 10^4$. Bubble e Insertion seguem $n$ na entrada ordenada (1,8 ms e 2,3 ms em $n = 10^4$) e $n^2$ nas demais. Na quase ordenada, o Insertion fica 33 a 38 vezes abaixo dos outros dois, mas paralelo a $n^2$. Na inversa, Bubble e Insertion levam mais que o dobro do tempo do Selection.

G3. O Selection faz exatamente $n(n-1)/2$ comparações, o Bubble praticamente o mesmo ($0{,}4998\,n^2$ em $n = 10^4$) e o Insertion cerca de $n^2/4$. Em movimentações, Bubble e Insertion coincidem em torno de $n^2/4$, e o Selection faz cerca de $n$ trocas.

G4. Merge e Quick aleatório acompanham a inclinação de $n\log n$; o Shellsort com Ciura é um pouco mais inclinado, mas fica bem abaixo de $n^{1{,}5}$. Em $n = 10^5$: 0,60 s (Quick), 0,72 s (Merge) e 0,91 s (Shellsort).

G5. Na entrada aleatória, a sequência de Ciura foi a melhor em tempo e em comparações (0,91 s e 2,53 milhões em $n = 10^5$), seguida de Knuth (1,43 s e 3,91 milhões) e de Shell (1,49 s e 4,32 milhões). Na entrada ordenada, Knuth ficou à frente (0,22 s, contra 0,28 s de Ciura e 0,35 s de Shell), e as comparações coincidiram exatamente com $\sum_h (n-h)$ sobre os *gaps* de cada sequência.

G6. O Counting Sort é linear com $k = 100$ e $k = n$ (0,37 s e 0,78 s em $n = 10^6$); com $k = n^{1{,}5}$, cresce com inclinação próxima de 1,5, passa o Merge a partir de $n = 10^4$ e foi interrompido em $n = 3\cdot10^5$ (22,3 s). O Radix varia bem menos com $k$ (0,55 s, 2,21 s e 2,99 s em $10^6$). O Bucket Sort é linear nas duas distribuições, com 1,47 s (uniforme) e 1,93 s (concentrada) em $10^6$. As referências levaram 8,3 s (Merge) e 7,2 s (Quick).

G7. Com pivô no último elemento, as entradas ordenada e inversa seguem $n^2$, com exatamente $n(n-1)/2$ comparações (1,54 s e 1,36 s em $n = 4\,000$). Os pivôs aleatório e mediana de 3 seguem $n\log n$ (de 9,9 ms a 22,8 ms); a mediana de 3 é a mais rápida na ordenada e a mais lenta na inversa. Com $k = 10$, as três estratégias praticamente coincidem (159 a 162 ms) e seguem $n^2$.

G8. Na entrada aleatória, o Insertion Sort já é o mais lento dos três em $n = 100$ (0,79 ms, contra 0,29 ms do Merge e 0,27 ms do Quick), sem cruzamento dentro da faixa. Na quase ordenada, ele é o mais rápido até $n = 1\,000$ e o mais lento em $n = 2\,000$, com cruzamento em torno de $n \approx 1\,250$ (Quick) e $n \approx 1\,600$ (Merge).

E6 (gráfico adicional). Com $k = 10^9$, o Radix fez 9, 4 e 2 passadas nas bases 10, 256 e 65 536, com 6,39 s, 2,98 s e 1,64 s em $n = 10^6$. Em $n = 10^4$, a base 256 foi a mais rápida (27,4 ms, contra 28,8 ms da base 65 536 e 53,5 ms da base 10).

T1. Os quadráticos ficaram entre 2,00 e 2,07, exceto Bubble e Insertion na entrada ordenada (1,03 e 0,99). Merge e Quick (aleatório e mediana de 3) ficaram entre 1,08 e 1,18, e o Shellsort entre 1,08 e 1,26. Os lineares ficaram entre 0,99 e 1,05 com $k$ fixo ou proporcional a $n$; o Counting com $k = n^{1{,}5}$ chegou a 1,51, o Radix com $k = n$ e $k = n^{1{,}5}$ a 1,09 e 1,12, e o Radix com base 65 536 ficou em 0,88. O Quick com pivô no último elemento chegou a 2,03 (ordenada) e 1,99 (inversa), e os três pivôs ficaram entre 1,89 e 1,94 com $k = 10$.

T2. As razões do Merge ($T/(n\log_2 n)$ entre $4{,}28\cdot10^{-7}$ e $4{,}51\cdot10^{-7}$) e do Insertion ($T/n^2$ entre $8{,}6\cdot10^{-8}$ e $9{,}4\cdot10^{-8}$) ficaram estáveis, e a razão de comparações do Selection chega a 0,5000. As razões de comparações do Merge e do Quick crescem devagar com $n$ (de 0,875 a 0,925 e de 1,08 a 1,24). No Shellsort com Ciura, $T/n^{1{,}5}$ cai cerca de cinco vezes na faixa. No Counting e no Bucket, o tempo por elemento cresce 35% e 44% entre $n = 10^5$ e $10^6$.

<a id="parteG"></a>
# Parte G — Discussão dos resultados ✍️

Responda a cada questão **com base nos seus dados**, citando o gráfico ou a tabela correspondente (ex.: "como se vê no G8…"). Respostas genéricas, sem referência aos experimentos, serão consideradas incompletas. Recomendamos de 1 a 3 parágrafos por questão.

### G.1 Quadráticos × $O(n \log n)$ para $n$ pequeno
*a) Em quais tamanhos e tipos de entrada algum algoritmo quadrático foi **mais rápido** que Merge Sort ou Quicksort? Estime o ponto de cruzamento $n_0$.*

Na entrada aleatória, nenhum: em $n = 100$, o menor tamanho do E1, o Insertion Sort já levou 0,79 ms, contra 0,29 ms do Merge e 0,27 ms do Quick (G8). O cruzamento fica abaixo de 100, e o modelo do item b o estima em $n_0 \approx 25$. Na entrada ordenada, Bubble e Insertion foram mais rápidos que todos os comparativos em todos os tamanhos (G2 e E1). Na quase ordenada, o Insertion venceu Merge e Quick até $n = 1\,000$ e perdeu em $n = 2\,000$; pela interpolação, $n_0 \approx 1\,250$ contra o Quick e $n_0 \approx 1\,600$ contra o Merge (G8).

*b) Explique o fenômeno usando as **constantes ocultas** da notação assintótica: modele $T_{ins}(n) \approx a\,n^2$ e $T_{merge}(n) \approx b\,n\log_2 n$, estime $a$ e $b$ a partir dos seus dados e calcule o $n_0$ previsto. Ele bate com o observado?*

Com os tempos de $n = 100$ na entrada aleatória, $a = 0{,}794\text{ ms}/100^2 \approx 7{,}9\cdot10^{-8}$ s e $b = 0{,}286\text{ ms}/(100\log_2 100) \approx 4{,}3\cdot10^{-7}$ s. Igualando, $n_0 = (b/a)\log_2 n_0 \approx 5{,}4\log_2 n_0$, que dá $n_0 \approx 25$ (23 com o Quick no lugar do Merge; entre 20 e 25 ajustando em $n = 200$ ou 500). Isso é coerente com o que se vê no G8, onde o cruzamento fica abaixo de 100, mas o E1 não tem tamanhos menores para confirmar o valor. As constantes vêm principalmente do número de operações: em Python instrumentado, cada comparação ou movimentação custou de 155 a 225 ns em todos os algoritmos, e em $n = 100$ o Insertion faz cerca de $n^2/2 \approx 5\,000$ operações, contra cerca de 1 300 do Merge, que ainda paga recursão e cópias para o vetor auxiliar.

*c) Com base nisso, justifique por que implementações reais (ex.: Timsort, *introsort*) usam Insertion Sort para subvetores pequenos. **(Opcional, +5 pontos bônus):** implemente um Merge Sort híbrido que usa `insertion_sort_intervalo` para subvetores com tamanho $\le L$, teste $L \in \{4, 8, 16, 32, 64\}$ e reporte o melhor $L$.*

Abaixo de algumas dezenas de elementos, o Insertion Sort faz menos trabalho por elemento e não tem recursão, e isso compensa o crescimento quadrático. Por isso Timsort e *introsort* trocam para ele nos subvetores pequenos. O Merge Sort híbrido (item opcional) não foi implementado.

### G.2 Impacto da escolha do pivô no Quicksort
*a) Descreva o comportamento de `quick_ultimo` em entradas ordenadas e inversas. O expoente $\alpha$ estimado é compatível com a recorrência $T(n) = T(n-1) + \Theta(n)$?*

Nas duas entradas, o `quick_ultimo` fez exatamente $n(n-1)/2$ comparações em todos os tamanhos (G7), com $\alpha = 2{,}03$ e $1{,}99$ no E3 e $2{,}03$ e $2{,}00$ no E5 (T1). É o que a recorrência prevê, $\Theta(n^2)$: na ordenada, o pivô é sempre o máximo, e uma das partes fica vazia; na inversa, o pivô é o mínimo, e a troca final leva o máximo para o fim do subvetor, de modo que as partições alternam entre mínimo e máximo, sempre com uma parte vazia. No E3, a combinação foi interrompida pelo limite de tempo depois de $n = 10^4$ (ordenada, 9,7 s) e em $n = 3\cdot10^4$ (inversa, 77,5 s numa única execução).

*b) Por que o pivô aleatório torna o pior caso **improvável**, mas não **impossível**? Qual a diferença entre "caso médio sobre as entradas" e "tempo esperado sobre as escolhas aleatórias do algoritmo"?*

O pior caso continua possível: basta o sorteio escolher um extremo do subvetor em praticamente todas as partições. Só que a probabilidade disso cai muito depressa com $n$. No caso médio sobre as entradas, o algoritmo é determinístico, e a média supõe uma distribuição para as entradas; uma entrada ruim continua sempre ruim. No tempo esperado sobre as escolhas do algoritmo, a garantia vale para qualquer entrada fixa, e a média é sobre os sorteios. As comparações do Quick aleatório ficaram a até 2,3% do valor esperado $2(n+1)H_n - 4n$ (153 695 contra 155 772 em $n = 10^4$). Como a semente do pivô é fixa (0), cada entrada recebe sempre a mesma sequência de pivôs.

*c) O que aconteceu com os três pivôs quando $k = 10$ (muitos repetidos)? Explique pela partição de Lomuto.*

As três estratégias degradaram igualmente para perto de $n^2$ ($\alpha$ entre 1,89 e 1,94, G7 e T1). A partição de Lomuto usada manda para a esquerda os elementos $\le$ pivô. Um subvetor formado só por cópias de um valor é então particionado com uma parte vazia a cada passo, e cada um dos $k$ valores, com cerca de $n/k$ cópias, custa cerca de $(n/k)^2/2$. O total é cerca de $n^2/(2k)$: 800 000 comparações para $n = 4\,000$, contra 814 mil a 826 mil medidas. Nenhuma escolha de pivô ajuda, porque dentro desses blocos todos os candidatos são iguais.

*d) Houve diferença de desvio padrão entre as estratégias? Por quê?*

Não de forma sistemática. As entradas ordenada e inversa são iguais em todas as *seeds*, e a semente do pivô aleatório é fixa, então as contagens tiveram desvio zero nas três estratégias. A variação de tempo nessas entradas, com coeficiente de 0,1% a 8%, é só ruído de medição. Com $k = 10$, a variação vem das diferentes entradas e é parecida nas três.

### G.3 Efeito da sequência de gaps no Shellsort
*a) Compare as três sequências em tempo e comparações. Qual foi a melhor? Quais foram os $\alpha$ estimados?*

Na entrada aleatória, a de Ciura foi a melhor em tempo e em comparações (0,91 s e 2,53 milhões em $n = 10^5$), seguida de Knuth (1,43 s e 3,91 milhões) e de Shell (1,49 s e 4,32 milhões), com $\alpha$ de 1,16, 1,22 e 1,23 (G5 e T1). Era o esperado, porque Ciura escolheu seus incrementos experimentalmente para minimizar as comparações em entradas aleatórias. Nas entradas ordenada e inversa, Knuth foi a mais rápida. Na ordenada, cada passada custa exatamente $n - h$ comparações, e vence quem tem menos *gaps*: Knuth, limitada a $h < n/3$, usa 10 em $n = 10^5$, contra 13 de Ciura e 16 de Shell.

*b) Relacione com os limites teóricos conhecidos (Shell: $\Theta(n^2)$ no pior caso; Knuth: $O(n^{3/2})$).*

O $\alpha$ de Knuth (1,22) fica abaixo de 1,5, compatível com o limite superior $O(n^{3/2})$. O de Shell (1,23) fica longe de 2, o que também é esperado: o pior caso dessa sequência exige $n$ potência de 2 e uma entrada construída para ele, e os experimentos usam entradas aleatórias em tamanhos que não são potências de 2. Na T2, $T/n^{1{,}5}$ do Shellsort com Ciura cai cerca de cinco vezes, o que põe seu crescimento abaixo de $n^{1{,}5}$ na faixa medida.

*c) O Shellsort foi competitivo com Merge Sort e Quicksort na sua faixa de $n$? E em entradas ordenadas/quase ordenadas?*

Foi. Na entrada aleatória, a versão de Ciura foi 1,3 a 1,5 vez mais lenta que Merge e Quick em $n = 10^5$ (G4), e a diferença tende a crescer, porque seu $\alpha$ é maior. Na entrada ordenada, Knuth e Ciura foram mais rápidas que Merge e Quick aleatório em todos os tamanhos do E3, e na inversa a partir de $n = 10^4$. Na quase ordenada, Ciura (0,67 s) ficou entre o Merge (0,63 s) e o Quick com mediana de 3 (0,98 s).

### G.4 Quando os algoritmos lineares são vantajosos?
*a) Com $k = 100$, $k = n$ e $k = n^{1,5}$, como se comportou o Counting Sort? Em que ponto o termo $k$ passa a dominar $\Theta(n + k)$? Houve problema de **memória**?*

Com $k = 100$ e $k = n$, o Counting foi linear ($\alpha$ = 1,00 e 1,05) e o mais rápido de todos (G6). Com $k = n^{1{,}5}$, o termo $k$ domina desde o primeiro tamanho: em $n = 10^3$, $k \approx 32\,n$ e o tempo já é 7,5 vezes o do caso $k = n$; o $\alpha$ medido (1,51) é o do próprio $k$. A combinação foi interrompida em $n = 3\cdot10^5$ pelo limite de tempo (22,3 s), e não por falta de memória. Pelas nossas contas, o vetor de contagem nesse tamanho, com $1{,}6\cdot10^8$ posições, ocupa alguns gigabytes. Com $n = 10^6$ ($k = 10^9$), seriam dezenas de gigabytes, acima dos 16 GB da máquina. A memória não foi medida.

*b) O Radix Sort foi afetado por $k$ da mesma forma? Relacione com $d = \lceil \log_b k \rceil$. Qual base foi melhor no E6 e por quê?*

Não. O custo do Radix cresce com o número de passadas $d$, ou seja, com $\log_b k$, e não com $k$. Em $n = 10^6$, os tempos de 0,55 s, 2,21 s e 2,99 s correspondem a 1, 3 e 4 passadas (G6). No E6, a base 65 536 foi a melhor nos tamanhos grandes (2 passadas, 1,64 s em $10^6$), e a base 10 a pior (9 passadas, 6,39 s). Em $n = 10^4$, a base 256 venceu, porque o vetor de contagem da base 65 536 é maior que o próprio $n$, e o termo $b$ de $\Theta(d(n+b))$ passa a pesar. A melhor base fica perto de $n$, como diz o resultado clássico.

*c) Compare o Bucket Sort com distribuição uniforme e concentrada. O resultado confirma a análise $\Theta(n)$ no caso médio e $\Theta(n^2)$ no pior caso?*

O caso uniforme confirma $\Theta(n)$ ($\alpha = 1{,}05$). O concentrado também foi linear ($\alpha = 1{,}01$), só que mais lento: 1,3 a 1,7 vez o tempo e 6 vezes as comparações (G6). A teoria também prevê isso, porque a distribuição concentrada põe 90% dos valores em 10% dos baldes, cerca de 9 elementos por balde, e esse número não cresce com $n$. O pior caso $\Theta(n^2)$ exige que uma fração crescente dos elementos caia num mesmo balde, e isso não foi testado.

*d) Em Python, os lineares foram de fato mais rápidos que Merge/Quick para $n$ grande? Discuta o papel das constantes, da alocação de memória e do fato de que "linear" pressupõe hipóteses sobre a entrada.*

Foram. Em $n = 10^6$, o Counting ($k = n$) levou 0,78 s, o Bucket uniforme 1,47 s e o Radix 2,21 s, contra 7,2 s do Quick e 8,3 s do Merge. A exceção foi o Counting com $k = n^{1{,}5}$, até 10 vezes mais lento que o Merge. A instrumentação pesa mais nos comparativos, que fazem uma chamada de função por operação, enquanto boa parte do trabalho dos lineares não é instrumentada. A alocação de memória pesa nos lineares quando $k$ ou $n$ crescem: o tempo por elemento do Counting e do Bucket subiu 35% e 44% entre $10^5$ e $10^6$ (T2). E a vantagem some quando a hipótese sobre a entrada falha, como no caso $k = n^{1{,}5}$.

*e) Por que não faz sentido comparar **número de comparações** entre algoritmos comparativos e não comparativos? Que métrica é mais justa?*

Counting e Radix fazem zero comparações, mas leem e escrevem cada elemento várias vezes e percorrem o vetor de contagem, e nada disso entra em `comp`. Métricas mais justas são o tempo, as movimentações, que os dois grupos têm, ou a contagem de todas as operações elementares, incluindo os acessos por índice.

### G.5 Teoria × prática (síntese)
*a) Os expoentes estimados (Tabela T1) ficaram próximos dos esperados? Aponte as maiores discrepâncias e proponha explicações (faixa de $n$ pequena, overhead do interpretador, cache, alocação de memória, instrumentação).*

Ficaram, na maior parte (T1): 2,00 a 2,07 nos quadráticos, cerca de 1 em Bubble e Insertion na entrada ordenada, 1,08 a 1,11 em Merge e Quick aleatório (o esperado para $n\log n$ nessa faixa é cerca de 1,1) e cerca de 1 nos lineares sob suas hipóteses. As maiores discrepâncias foram estas:
- Na entrada quase ordenada, Bubble e Insertion tiveram $\alpha \approx 2$. O gerador troca cerca de $n/100$ pares sorteados, a uma distância média de $n/3$, o que cria cerca de $n^2/150$ inversões. As 652 828 comparações do Insertion em $n = 10^4$ (cerca de $n^2/153$) confirmam a conta.
- O Radix ficou acima de 1 com $k = n$ e $k = n^{1{,}5}$ (1,09 e 1,12), porque o número de passadas cresce com $n$, e abaixo de 1 com base 65 536 (0,88), porque até $n = 10^4$ o vetor de contagem é maior que $n$.
- O tempo por elemento do Counting e do Bucket cresceu a partir de $10^5$ (T2). A explicação mais provável é a memória e o cache: o processador tem 25 MB de cache L3, e uma lista de $10^6$ referências já ocupa 8 MB. Isso não foi medido.

*b) Por que **contagens de operações** são mais estáveis que **tempo** entre seeds? Qual das duas métricas é melhor para validar a análise assintótica? E para decidir qual algoritmo usar em um sistema real?*

Para uma dada entrada, as contagens são determinísticas. Tiveram desvio zero nas entradas ordenada e inversa e deram valores idênticos nas duas execuções completas que fizemos. O tempo inclui ruído do sistema operacional, do cache e da memória. Para validar a análise assintótica, as contagens são a melhor métrica: reproduziram exatamente $n(n-1)/2$ e $\sum_h (n-h)$, e quase exatamente $n^2/(2k)$ e $2(n+1)H_n - 4n$. Para escolher um algoritmo num sistema real, o tempo informa mais, porque inclui o custo de cada operação e o trabalho não contado. Em $n = 10^5$, por exemplo, o Quick aleatório fez 34% mais comparações que o Merge e ainda assim foi 16% mais rápido, porque moveu menos elementos e não copia para um vetor auxiliar.

*c) Monte uma tabela de **recomendação**: para cada cenário — (i) $n$ pequeno; (ii) dados quase ordenados; (iii) necessidade de estabilidade; (iv) memória extra restrita; (v) inteiros em intervalo pequeno; (vi) pior caso garantido — qual algoritmo você escolheria e por quê?*

| Cenário | Escolha | Justificativa |
|---|---|---|
| (i) $n$ pequeno | Insertion Sort até algumas dezenas de elementos; Shellsort (Ciura) por volta de $n = 100$ | $n_0$ estimado entre 23 e 25; em $n = 100$, o Shellsort foi o comparativo mais rápido (E1) |
| (ii) Dados quase ordenados | Insertion Sort se as inversões forem poucas e locais; senão, Quick aleatório ou Shellsort | linear na entrada ordenada; com inversões a longa distância, só compensou até $n \approx 10^3$ (G8) |
| (iii) Estabilidade | Merge Sort; Counting ou Radix para inteiros | estáveis por construção (testes da Parte D) |
| (iv) Pouca memória extra | Shellsort ou Quicksort aleatório | *in-place*, pilha $O(\log n)$ no Quick; competitivos com o Merge (G4) |
| (v) Inteiros em intervalo pequeno | Counting Sort ($k \lesssim n$); Radix com base grande para $k$ grande | os mais rápidos do trabalho; Radix pouco sensível a $k$ (G6) |
| (vi) Pior caso garantido | Merge Sort | $\Theta(n\log n)$ em qualquer entrada; menor variação entre tipos (de 0,56 a 0,72 s em $10^5$) |

<a id="parteH"></a>
# Parte H — Relatório ✍️

O relatório é escrito **neste mesmo notebook**, nas células abaixo, e reaproveita o conteúdo das partes anteriores (você pode referenciar tabelas e gráficos já gerados, sem duplicá-los). Extensão recomendada do texto próprio do relatório: **equivalente a 6 a 12 páginas**, sem contar código.

### Requisitos de cada seção

| Seção | O que deve conter |
|---|---|
| **1. Introdução** | Contexto e importância da ordenação; objetivo do trabalho; perguntas que o trabalho responde; organização do relatório. |
| **2. Fundamentação teórica** | Descrição sucinta de cada algoritmo; tabela consolidada de complexidades (Parte E); recorrências e aplicação do Teorema Mestre; limite inferior $\Omega(n \log n)$ para ordenação por comparação e as hipóteses dos lineares. |
| **3. Metodologia** | Ambiente (CPU, RAM, sistema operacional, versão do Python — use a célula de apoio abaixo); decisões de implementação (partição usada, base do Radix, gaps, o que foi contado como comparação/troca); geração das entradas; faixas de $n$ e justificativa; seeds e número de repetições; como o tempo foi medido; ameaças à validade (ruído, overhead da instrumentação). Deve ser detalhada o suficiente para que **outra pessoa reproduza** os experimentos. |
| **4. Resultados** | Gráficos e tabelas obrigatórios, com legendas numeradas e descrição objetiva do que mostram. |
| **5. Análise crítica** | A discussão da Parte G organizada em texto corrido; confronto com as previsões da Parte E; limitações do estudo. |
| **6. Conclusões** | Principais achados em poucas frases; recomendações práticas; possíveis trabalhos futuros (ex.: Heapsort, Timsort, versões iterativas, outras linguagens). |
| **Referências** | Livros, artigos e sites consultados, em formato ABNT ou similar. Sugestões: CORMEN et al., *Algoritmos: Teoria e Prática*; ZIVIANI, *Projeto de Algoritmos*; SEDGEWICK & WAYNE, *Algorithms*; CIURA, M. *Best Increments for the Average Case of Shellsort* (2001). |